In [ ]:
!pip -q install pandas geopandas folium openpyxl

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster, HeatMap
from google.colab import files

In [ ]:
uploaded = files.upload()

csv_path = list(uploaded.keys())[0]
print("Uploaded file:", csv_path)

df = pd.read_csv(csv_path)

print("Rows:", len(df))
print(df.head())
print(df.columns)

In [ ]:
# Ensure coordinates are numeric
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# Drop records without usable coordinates
df = df.dropna(subset=["lat", "lon"]).copy()

# Create a display label
df["site_label"] = df["name"].fillna("Unnamed data centre")
df["operator_label"] = df["operator"].fillna("Unknown operator")

# Check country counts
country_counts = df["country"].value_counts().reset_index()
country_counts.columns = ["country", "count"]

country_counts

In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

gdf.head()

In [ ]:
gdf.to_file("2Europe_DC_2024_points.geojson", driver="GeoJSON")
gdf.to_file("2Europe_DC_2024_points.gpkg", layer="data_centres", driver="GPKG")

print("Saved:")
print("2Europe_DC_2024_points.geojson")
print("2Europe_DC_2024_points.gpkg")

In [ ]:
# Centre map roughly over Europe
m = folium.Map(
    location=[53, 10],
    zoom_start=4,
    tiles="CartoDB positron"
)

marker_cluster = MarkerCluster(name="Data centres").add_to(m)

for _, row in df.iterrows():
    popup_html = f"""
    <b>{row['name']}</b><br>
    Country: {row['country']}<br>
    Operator: {row['operator']}<br>
    OSM: {row['osm_type']}/{row['osm_id']}<br>
    Confirmed as of: {row['confirmed_as_of']}
    """

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=row["name"],
        fill=True
    ).add_to(marker_cluster)

folium.LayerControl().add_to(m)

m

In [ ]:
# Centre map roughly over Europe
m = folium.Map(
    location=[53, 10],
    zoom_start=4,
    tiles=None
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite imagery",
    overlay=False,
    control=True
).add_to(m)

marker_cluster = MarkerCluster(name="Data centres").add_to(m)

for _, row in df.iterrows():
    popup_html = f"""
    <b>{row['name']}</b><br>
    Country: {row['country']}<br>
    Operator: {row['operator']}<br>
    OSM: {row['osm_type']}/{row['osm_id']}<br>
    Confirmed as of: {row['confirmed_as_of']}
    """

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=row["name"],
        fill=True
    ).add_to(marker_cluster)

folium.LayerControl().add_to(m)

m

In [ ]:
#m.save("Buildings_Europe_DC_2024_interactive_map.html")

#files.download("Buildings_Europe_DC_2024_interactive_map.html")

In [ ]:
m_heat = folium.Map(
    location=[53, 10],
    zoom_start=4,
    tiles="CartoDB positron"
)

heat_data = df[["lat", "lon"]].values.tolist()

HeatMap(
    heat_data,
    radius=18,
    blur=20,
    name="Data centre density"
).add_to(m_heat)

folium.LayerControl().add_to(m_heat)

m_heat

In [ ]:
import pandas as pd

df = pd.read_csv("2Europe_DC_2024.csv")

df["facility_id"] = (
    df["country"].astype(str)
    + "_"
    + df["osm_type"].astype(str)
    + "_"
    + df["osm_id"].astype(str)
)

# Optional: move facility_id to the first column
cols = ["facility_id"] + [c for c in df.columns if c != "facility_id"]
df = df[cols]

df.to_csv("Europe_DC_2024_with_facility_id.csv", index=False)

df.head()

In [ ]:
#!pip -q install pandas geopandas shapely folium openpyxl

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [ ]:
df = pd.read_csv("Europe_DC_2024_with_facility_id.csv")

# Remove old geometry column if it exists
df = df.drop(columns=["geometry"], errors="ignore")

# Make sure coordinates are numeric
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# Remove rows without coordinates
df = df.dropna(subset=["lat", "lon"]).copy()

print("Rows:", len(df))
df.head()

In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

gdf.to_file("Europe_DC_2024_points_with_facility_id.geojson", driver="GeoJSON")
gdf.to_file("Europe_DC_2024_points_with_facility_id.gpkg", layer="data_centres", driver="GPKG")

print("Saved point files with facility_id.")

# Buffers

In [ ]:
# Project to European CRS for metre-based buffers
gdf_3035 = gdf.to_crs("EPSG:3035")

buffer_distances_km = [1, 2.5, 5, 10]

all_buffers = []

for km in buffer_distances_km:
    buffer_gdf = gdf_3035.copy()
    buffer_gdf["buffer_km"] = km
    buffer_gdf["geometry"] = buffer_gdf.geometry.buffer(km * 1000)

    # Keep useful join fields
    keep_cols = [
        "facility_id",
        "country",
        "operator",
        "buffer_km",
        "geometry"
    ]

    buffer_gdf = buffer_gdf[keep_cols]

    label = str(km).replace(".", "p")

    # Save projected version for analysis
    buffer_gdf.to_file(
        f"Europe_DC_2024_buffer_{label}km.gpkg",
        layer=f"buffer_{label}km",
        driver="GPKG"
    )

    # Save WGS84 version for viewing/mapping
    buffer_gdf.to_crs("EPSG:4326").to_file(
        f"Europe_DC_2024_buffer_{label}km.geojson",
        driver="GeoJSON"
    )

    all_buffers.append(buffer_gdf)

    print(f"Saved {km} km buffer")

In [ ]:
combined_buffers = pd.concat(all_buffers, ignore_index=True)
combined_buffers = gpd.GeoDataFrame(
    combined_buffers,
    geometry="geometry",
    crs="EPSG:3035"
)

combined_buffers.to_file(
    "Europe_DC_2024_all_buffers.gpkg",
    layer="all_buffers",
    driver="GPKG"
)

combined_buffers.to_crs("EPSG:4326").to_file(
    "Europe_DC_2024_all_buffers.geojson",
    driver="GeoJSON"
)

print("Saved combined buffer files.")

In [ ]:
combined_buffers.head()

In [ ]:
import geopandas as gpd
import pandas as pd

buffers = gpd.read_file("Europe_DC_2024_all_buffers.geojson")

print("Rows:", len(buffers))
print("Unique facilities:", buffers["facility_id"].nunique())
print("Buffer distances:", sorted(buffers["buffer_km"].unique()))

buffers.head()

In [ ]:
buffer_check = buffers.groupby("facility_id")["buffer_km"].nunique().reset_index()
buffer_check.columns = ["facility_id", "number_of_buffers"]

print(buffer_check["number_of_buffers"].value_counts())

missing_buffers = buffer_check[buffer_check["number_of_buffers"] != 4]
missing_buffers.head()

In [ ]:
indicator_table_long = buffers[[
    "facility_id",
    "country",
    "operator",
    "buffer_km"
]].copy()

indicator_table_long = indicator_table_long.sort_values(
    ["facility_id", "buffer_km"]
).reset_index(drop=True)

indicator_table_long.head(10)

In [ ]:
indicator_table_long.to_csv("Europe_DC_2024_indicator_table_long_blank.csv", index=False)
indicator_table_long.to_excel("Europe_DC_2024_indicator_table_long_blank.xlsx", index=False)

print("Saved blank indicator table.")

In [ ]:
facility_table = pd.read_csv("Europe_DC_2024_with_facility_id.csv")

facility_indicator_table = facility_table[[
    "facility_id",
    "country",
    "operator",
    "lat",
    "lon"
]].copy()

facility_indicator_table.to_csv("Europe_DC_2024_indicator_table_wide_blank.csv", index=False)
facility_indicator_table.to_excel("Europe_DC_2024_indicator_table_wide_blank.xlsx", index=False)

facility_indicator_table.head()

In [ ]:
buffer_check = indicator_table_long.groupby("facility_id")["buffer_km"].nunique().reset_index()
buffer_check.columns = ["facility_id", "number_of_buffers"]

print(buffer_check["number_of_buffers"].value_counts())

missing = buffer_check[buffer_check["number_of_buffers"] != 4]
missing

In [ ]:
print("Total buffer rows:", len(indicator_table_long))
print("Unique facilities:", indicator_table_long["facility_id"].nunique())
print("Expected rows:", indicator_table_long["facility_id"].nunique() * 4)

# Population

In [ ]:
!pip -q install earthengine-api geemap geopandas pandas openpyxl

import pandas as pd
import geopandas as gpd
import numpy as np
import ee
import geemap
from pathlib import Path
from google.colab import drive

In [ ]:
ee.Authenticate()
ee.Initialize(project="YOUR-EARTH-ENGINE-PROJECT")

In [ ]:
facility = pd.read_csv("Europe_DC_2024_with_facility_id.csv")

# Remove old geometry text column if it exists
facility = facility.drop(columns=["geometry"], errors="ignore")

# Make coordinates numeric
facility["lat"] = pd.to_numeric(facility["lat"], errors="coerce")
facility["lon"] = pd.to_numeric(facility["lon"], errors="coerce")

# Remove invalid coordinates
facility = facility.dropna(subset=["lat", "lon"]).copy()

# Keep only the fields needed for Earth Engine
facility_ee_input = facility[[
    "facility_id",
    "country",
    "operator",
    "lat",
    "lon"
]].copy()

facility_ee_input["operator"] = facility_ee_input["operator"].fillna("Unknown").astype(str)

print("Facilities:", len(facility_ee_input))
facility_ee_input.head()

In [ ]:
points_gdf = gpd.GeoDataFrame(
    facility_ee_input,
    geometry=gpd.points_from_xy(
        facility_ee_input["lon"],
        facility_ee_input["lat"]
    ),
    crs="EPSG:4326"
)

try:
    points_ee = geemap.geopandas_to_ee(points_gdf)
except AttributeError:
    points_ee = geemap.gdf_to_ee(points_gdf)

print("Point FeatureCollection created.")
print("Number of points:", points_ee.size().getInfo())

In [ ]:
buffer_distances_m = ee.List([1000, 2500, 5000, 10000])

def make_buffers_for_distance(distance_m):
    distance_m = ee.Number(distance_m)
    buffer_km = distance_m.divide(1000)

    return points_ee.map(
        lambda feature: ee.Feature(
            feature.geometry().buffer(distance_m, maxError=50),
            {
                "facility_id": feature.get("facility_id"),
                "country": feature.get("country"),
                "operator": feature.get("operator"),
                "buffer_m": distance_m,
                "buffer_km": buffer_km
            }
        )
    )

buffers_ee = ee.FeatureCollection(
    buffer_distances_m.map(make_buffers_for_distance)
).flatten()

print("Number of Earth Engine buffer features:", buffers_ee.size().getInfo())

In [ ]:
# GHSL population, nearest epoch to the 2024 data-centre inventory
pop_raw = ee.Image("JRC/GHSL/P2023A/GHS_POP/2025").select("population_count")

# Correct invalid negative/no-data values
# Population cannot be negative, so any negative cells are forced to 0
pop = pop_raw.unmask(0).max(ee.Image.constant(0)).rename("population_count")

# Sum population inside each buffer
pop_stats = pop.reduceRegions(
    collection=buffers_ee,
    reducer=ee.Reducer.sum(),
    scale=100,
    tileScale=8
)

In [ ]:
def make_population_table_row(feature):
    return ee.Feature(None, {
        "facility_id": feature.get("facility_id"),
        "country": feature.get("country"),
        "operator": feature.get("operator"),
        "buffer_m": feature.get("buffer_m"),
        "buffer_km": feature.get("buffer_km"),
        "pop_sum": feature.get("sum")
    })

pop_table = pop_stats.map(make_population_table_row)

In [ ]:
task_pop_final = ee.batch.Export.table.toDrive(
    collection=pop_table,
    description="GHSL_population_by_buffer_NONNEG_FINAL",
    fileNamePrefix="GHSL_population_by_buffer_NONNEG_FINAL",
    fileFormat="CSV",
    selectors=[
        "facility_id",
        "country",
        "operator",
        "buffer_m",
        "buffer_km",
        "pop_sum"
    ]
)

task_pop_final.start()

print("Corrected population export started.")

In [ ]:
task_pop_final.status()

In [ ]:
drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive")

matches = sorted(
    drive_root.rglob("GHSL_population_by_buffer_NONNEG_FINAL*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

matches

In [ ]:
pop_file = matches[0]
print("Using corrected population file:", pop_file)

pop_long = pd.read_csv(pop_file)

pop_long["buffer_m"] = pd.to_numeric(pop_long["buffer_m"], errors="coerce")
pop_long["buffer_km"] = pd.to_numeric(pop_long["buffer_km"], errors="coerce")
pop_long["pop_sum"] = pd.to_numeric(pop_long["pop_sum"], errors="coerce")

print(pop_long.shape)
pop_long.head()

In [ ]:
print("Negative population rows:", (pop_long["pop_sum"] < 0).sum())

print("\nRows by buffer:")
print(pop_long["buffer_m"].value_counts().sort_index())

print("\nMinimum population value:")
print(pop_long["pop_sum"].min())

In [ ]:
pop_wide = pop_long.pivot_table(
    index="facility_id",
    columns="buffer_m",
    values="pop_sum",
    aggfunc="first"
).reset_index()

pop_wide = pop_wide.rename(columns={
    1000: "pop_1km",
    2500: "pop_2p5km",
    5000: "pop_5km",
    10000: "pop_10km"
})

pop_wide.head()

In [ ]:
facility_with_pop = facility.merge(
    pop_wide,
    on="facility_id",
    how="left"
)

print("Rows:", len(facility_with_pop))

print("\nMissing population values:")
print(facility_with_pop[[
    "pop_1km",
    "pop_2p5km",
    "pop_5km",
    "pop_10km"
]].isna().sum())

In [ ]:
eps = 1e-6

facility_with_pop["pop_monotonic_ok"] = (
    (facility_with_pop["pop_1km"] <= facility_with_pop["pop_2p5km"] + eps) &
    (facility_with_pop["pop_2p5km"] <= facility_with_pop["pop_5km"] + eps) &
    (facility_with_pop["pop_5km"] <= facility_with_pop["pop_10km"] + eps)
)

facility_with_pop["pop_monotonic_ok"].value_counts()

In [ ]:
facility_with_pop["pop_density_1km"] = facility_with_pop["pop_1km"] / (np.pi * 1**2)
facility_with_pop["pop_density_2p5km"] = facility_with_pop["pop_2p5km"] / (np.pi * 2.5**2)
facility_with_pop["pop_density_5km"] = facility_with_pop["pop_5km"] / (np.pi * 5**2)
facility_with_pop["pop_density_10km"] = facility_with_pop["pop_10km"] / (np.pi * 10**2)

facility_with_pop[[
    "facility_id",
    "pop_1km",
    "pop_2p5km",
    "pop_5km",
    "pop_10km",
    "pop_density_1km",
    "pop_density_2p5km",
    "pop_density_5km",
    "pop_density_10km",
    "pop_monotonic_ok"
]].head()

In [ ]:
pop_wide.to_csv(
    "Europe_DC_2024_population_indicators_FINAL.csv",
    index=False
)

pop_wide.to_excel(
    "Europe_DC_2024_population_indicators_FINAL.xlsx",
    index=False
)

facility_with_pop.to_csv(
    "Europe_DC_2024_with_population_indicators_FINAL.csv",
    index=False
)

facility_with_pop.to_excel(
    "Europe_DC_2024_with_population_indicators_FINAL.xlsx",
    index=False
)

print("Saved final corrected population outputs.")

In [ ]:
pop_cols = ["pop_1km", "pop_2p5km", "pop_5km", "pop_10km"]

print("Negative values:")
print((facility_with_pop[pop_cols] < 0).sum())

print("\nMissing values:")
print(facility_with_pop[pop_cols].isna().sum())

print("\nMonotonic check:")
print(facility_with_pop["pop_monotonic_ok"].value_counts())

print("\nRows:", len(facility_with_pop))

In [ ]:
facility_with_pop.to_csv(
    "Europe_DC_2024_with_population_indicators_FINAL.csv",
    index=False
)

facility_with_pop.to_excel(
    "Europe_DC_2024_with_population_indicators_FINAL.xlsx",
    index=False
)

print("Saved final population indicator files.")

# GHSL built-up

In [ ]:
# GHSL built-up surface, nearest epoch to the 2024 data-centre inventory
built_raw = ee.Image("JRC/GHSL/P2023A/GHS_BUILT_S/2025").select("built_surface")

# Built-up surface is measured in m² per 100 m grid cell.
# Force no-data / invalid negative cells to zero.
built = built_raw.unmask(0).max(ee.Image.constant(0)).rename("built_surface")

In [ ]:
built_stats = built.reduceRegions(
    collection=buffers_ee,
    reducer=ee.Reducer.sum(),
    scale=100,
    tileScale=8
)

In [ ]:
def make_built_table_row(feature):
    return ee.Feature(None, {
        "facility_id": feature.get("facility_id"),
        "country": feature.get("country"),
        "operator": feature.get("operator"),
        "buffer_m": feature.get("buffer_m"),
        "buffer_km": feature.get("buffer_km"),
        "built_sum_m2": feature.get("sum")
    })

built_table = built_stats.map(make_built_table_row)

In [ ]:
task_built = ee.batch.Export.table.toDrive(
    collection=built_table,
    description="GHSL_built_surface_by_buffer_FINAL",
    fileNamePrefix="GHSL_built_surface_by_buffer_FINAL",
    fileFormat="CSV",
    selectors=[
        "facility_id",
        "country",
        "operator",
        "buffer_m",
        "buffer_km",
        "built_sum_m2"
    ]
)

task_built.start()

print("Built-up surface export started.")

In [ ]:
task_built.status()

In [ ]:
drive_root = Path("/content/drive/MyDrive")

matches = sorted(
    drive_root.rglob("GHSL_built_surface_by_buffer_FINAL*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

matches

In [ ]:
built_file = matches[0]
print("Using built-up file:", built_file)

built_long = pd.read_csv(built_file)

built_long["buffer_m"] = pd.to_numeric(built_long["buffer_m"], errors="coerce")
built_long["buffer_km"] = pd.to_numeric(built_long["buffer_km"], errors="coerce")
built_long["built_sum_m2"] = pd.to_numeric(built_long["built_sum_m2"], errors="coerce")

print(built_long.shape)
built_long.head()

In [ ]:
print("Negative built-up rows:", (built_long["built_sum_m2"] < 0).sum())

print("\nRows by buffer:")
print(built_long["buffer_m"].value_counts().sort_index())

print("\nMinimum built-up value:")
print(built_long["built_sum_m2"].min())

print("\nMaximum built-up value:")
print(built_long["built_sum_m2"].max())

In [ ]:
import numpy as np

# Buffer area in m²
built_long["buffer_area_m2"] = np.pi * (built_long["buffer_m"] ** 2)

# Percentage of buffer area that is built-up
built_long["built_pct"] = (
    built_long["built_sum_m2"] / built_long["buffer_area_m2"]
) * 100

built_long.head()

In [ ]:
built_long[["built_sum_m2", "built_pct"]].describe()

In [ ]:
built_wide_sum = built_long.pivot_table(
    index="facility_id",
    columns="buffer_m",
    values="built_sum_m2",
    aggfunc="first"
).reset_index()

built_wide_sum = built_wide_sum.rename(columns={
    1000: "built_m2_1km",
    2500: "built_m2_2p5km",
    5000: "built_m2_5km",
    10000: "built_m2_10km"
})

built_wide_pct = built_long.pivot_table(
    index="facility_id",
    columns="buffer_m",
    values="built_pct",
    aggfunc="first"
).reset_index()

built_wide_pct = built_wide_pct.rename(columns={
    1000: "built_pct_1km",
    2500: "built_pct_2p5km",
    5000: "built_pct_5km",
    10000: "built_pct_10km"
})

built_wide = built_wide_sum.merge(
    built_wide_pct,
    on="facility_id",
    how="left"
)

built_wide.head()

In [ ]:
facility_with_pop = pd.read_csv("Europe_DC_2024_with_population_indicators_FINAL.csv")

facility_with_pop_built = facility_with_pop.merge(
    built_wide,
    on="facility_id",
    how="left"
)

print("Rows:", len(facility_with_pop_built))

print("\nMissing built-up values:")
print(facility_with_pop_built[[
    "built_m2_1km",
    "built_m2_2p5km",
    "built_m2_5km",
    "built_m2_10km",
    "built_pct_1km",
    "built_pct_2p5km",
    "built_pct_5km",
    "built_pct_10km"
]].isna().sum())

In [ ]:
built_cols = [
    "built_m2_1km",
    "built_m2_2p5km",
    "built_m2_5km",
    "built_m2_10km"
]

pct_cols = [
    "built_pct_1km",
    "built_pct_2p5km",
    "built_pct_5km",
    "built_pct_10km"
]

print("Negative built-up values:")
print((facility_with_pop_built[built_cols] < 0).sum())

print("\nMissing built-up values:")
print(facility_with_pop_built[built_cols + pct_cols].isna().sum())

print("\nBuilt-up percentage summary:")
print(facility_with_pop_built[pct_cols].describe())

In [ ]:
built_wide.to_csv(
    "Europe_DC_2024_builtup_indicators_FINAL.csv",
    index=False
)

built_wide.to_excel(
    "Europe_DC_2024_builtup_indicators_FINAL.xlsx",
    index=False
)

facility_with_pop_built.to_csv(
    "Europe_DC_2024_with_population_builtup_indicators_FINAL.csv",
    index=False
)

facility_with_pop_built.to_excel(
    "Europe_DC_2024_with_population_builtup_indicators_FINAL.xlsx",
    index=False
)

print("Saved final population + built-up indicator files.")

In [ ]:
# Final built-up completion checks

import pandas as pd
from pathlib import Path

final_file = Path("Europe_DC_2024_with_population_builtup_indicators_FINAL.csv")

print("Final file exists:", final_file.exists())

facility_with_pop_built = pd.read_csv(final_file)

built_cols = [
    "built_m2_1km",
    "built_m2_2p5km",
    "built_m2_5km",
    "built_m2_10km"
]

pct_cols = [
    "built_pct_1km",
    "built_pct_2p5km",
    "built_pct_5km",
    "built_pct_10km"
]

print("\nRows:", len(facility_with_pop_built))

print("\nMissing built-up values:")
print(facility_with_pop_built[built_cols + pct_cols].isna().sum())

print("\nNegative built-up values:")
print((facility_with_pop_built[built_cols] < 0).sum())

print("\nBuilt-up percentage summary:")
print(facility_with_pop_built[pct_cols].describe())

In [ ]:
# Indicator QA: population and built-up

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

master = pd.read_csv("Europe_DC_2024_with_population_builtup_indicators_FINAL.csv")

print("Rows:", len(master))
print("Columns:", master.columns.tolist())

In [ ]:
indicator_cols = [
    "pop_1km",
    "pop_2p5km",
    "pop_5km",
    "pop_10km",
    "pop_density_1km",
    "pop_density_2p5km",
    "pop_density_5km",
    "pop_density_10km",
    "built_m2_1km",
    "built_m2_2p5km",
    "built_m2_5km",
    "built_m2_10km",
    "built_pct_1km",
    "built_pct_2p5km",
    "built_pct_5km",
    "built_pct_10km"
]

missing_summary = master[indicator_cols].isna().sum().reset_index()
missing_summary.columns = ["variable", "missing_count"]

missing_summary

In [ ]:
negative_summary = (master[indicator_cols] < 0).sum().reset_index()
negative_summary.columns = ["variable", "negative_count"]

negative_summary

In [ ]:
master[indicator_cols].describe().T

In [ ]:
plot_cols = [
    "pop_density_2p5km",
    "built_pct_2p5km",
    "pop_2p5km",
    "built_m2_2p5km"
]

for col in plot_cols:
    plt.figure(figsize=(7, 4))
    master[col].hist(bins=40)
    plt.title(col)
    plt.xlabel(col)
    plt.ylabel("Number of data centres")
    plt.show()

In [ ]:
too_high_built = master[
    (master["built_pct_1km"] > 100) |
    (master["built_pct_2p5km"] > 100) |
    (master["built_pct_5km"] > 100) |
    (master["built_pct_10km"] > 100)
]

print("Facilities with built_pct above 100:", len(too_high_built))

too_high_built[[
    "facility_id",
    "country",
    "name",
    "lat",
    "lon",
    "built_pct_1km",
    "built_pct_2p5km",
    "built_pct_5km",
    "built_pct_10km"
]].head(20)

In [ ]:
master = pd.read_csv("Europe_DC_2024_with_population_builtup_indicators_FINAL.csv")

indicator_cols = [
    "pop_1km",
    "pop_2p5km",
    "pop_5km",
    "pop_10km",
    "pop_density_1km",
    "pop_density_2p5km",
    "pop_density_5km",
    "pop_density_10km",
    "built_m2_1km",
    "built_m2_2p5km",
    "built_m2_5km",
    "built_m2_10km",
    "built_pct_1km",
    "built_pct_2p5km",
    "built_pct_5km",
    "built_pct_10km"
]

print("Rows:", len(master))

print("\nMissing values:")
print(master[indicator_cols].isna().sum())

print("\nNegative values:")
print((master[indicator_cols] < 0).sum())

print("\nSummary:")
display(master[indicator_cols].describe().T)

In [ ]:
built_monotonic_ok = (
    (master["built_m2_1km"] <= master["built_m2_2p5km"]) &
    (master["built_m2_2p5km"] <= master["built_m2_5km"]) &
    (master["built_m2_5km"] <= master["built_m2_10km"])
)

built_monotonic_ok.value_counts()

In [ ]:
qa_summary = master[
    [
        "pop_1km", "pop_2p5km", "pop_5km", "pop_10km",
        "pop_density_1km", "pop_density_2p5km", "pop_density_5km", "pop_density_10km",
        "built_m2_1km", "built_m2_2p5km", "built_m2_5km", "built_m2_10km",
        "built_pct_1km", "built_pct_2p5km", "built_pct_5km", "built_pct_10km"
    ]
].describe().T

qa_summary.to_csv("QA_population_builtup_summary.csv")
qa_summary.to_excel("QA_population_builtup_summary.xlsx")

print("Saved QA summary.")

# Creating buffer File for data download

In [ ]:
import geopandas as gpd
from pathlib import Path

# Load all buffers
buffers = gpd.read_file("Europe_DC_2024_all_buffers.gpkg", layer="all_buffers")

print("Rows:", len(buffers))
print("Buffer distances:", sorted(buffers["buffer_km"].unique()))
print("CRS:", buffers.crs)

In [ ]:
buffers_10km = buffers[buffers["buffer_km"] == 10].copy()

print("10 km buffers:", len(buffers_10km))
print("Unique facilities:", buffers_10km["facility_id"].nunique())

In [ ]:
# Dissolve all 10 km buffers into one geometry
aoi = buffers_10km.dissolve()

# Simplify slightly to reduce file size
# This is in EPSG:3035 metres, so tolerance=100 means 100 m simplification
aoi["geometry"] = aoi.geometry.simplify(
    tolerance=100,
    preserve_topology=True
)

# Convert to WGS84 for upload to the Copernicus Data Viewer
aoi_wgs84 = aoi.to_crs("EPSG:4326")

# Keep only geometry
aoi_wgs84 = aoi_wgs84[["geometry"]].copy()
aoi_wgs84["aoi_name"] = "Europe_DC_10km_buffer_union"

aoi_wgs84.to_file(
    "Europe_DC_10km_buffer_union_AOI.geojson",
    driver="GeoJSON"
)

print("Saved AOI file:")
print("Europe_DC_10km_buffer_union_AOI.geojson")

In [ ]:
from google.colab import files
files.download("Europe_DC_10km_buffer_union_AOI.geojson")

# Splitting buffer file (don't need)

In [ ]:
import geopandas as gpd
from pathlib import Path
from google.colab import files

buffers = gpd.read_file("Europe_DC_2024_all_buffers.gpkg", layer="all_buffers")

# Keep only 10 km buffers because these contain all smaller buffers too
buffers_10km = buffers[buffers["buffer_km"] == 10].copy()

regions = {
    "west_europe": ["GB", "IE", "FR", "BE", "NL"],
    "central_europe": ["DE", "AT", "CH", "CZ", "PL"],
    "north_europe": ["DK", "SE", "NO", "FI"],
    "south_europe": ["ES", "PT", "IT", "GR"],
    "east_europe": ["HU", "RO"]
}

output_dir = Path("copernicus_regional_aois")
output_dir.mkdir(exist_ok=True)

for region_name, countries in regions.items():
    subset = buffers_10km[buffers_10km["country"].isin(countries)].copy()

    print(region_name, "facilities:", subset["facility_id"].nunique())

    if len(subset) == 0:
        continue

    # Dissolve all buffers in that region
    aoi = subset.dissolve()

    # Simplify in EPSG:3035 metres to reduce file size
    aoi["geometry"] = aoi.geometry.simplify(
        tolerance=100,
        preserve_topology=True
    )

    # Convert to WGS84 for Copernicus upload
    aoi_wgs84 = aoi.to_crs("EPSG:4326")
    aoi_wgs84 = aoi_wgs84[["geometry"]].copy()
    aoi_wgs84["aoi_name"] = region_name

    out_file = output_dir / f"{region_name}_10km_buffer_AOI.geojson"
    aoi_wgs84.to_file(out_file, driver="GeoJSON")

    print("Saved:", out_file)

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "copernicus_regional_aois",
    "zip",
    output_dir
)

files.download(zip_path)

# Land cover

In [ ]:
"""# Dynamic World 2024 land-cover indicators"""

!pip -q install earthengine-api geemap geopandas pandas openpyxl

import pandas as pd
import geopandas as gpd
import numpy as np
import ee
import geemap
from pathlib import Path
from google.colab import drive

In [ ]:
ee.Authenticate()
ee.Initialize(project="YOUR-EARTH-ENGINE-PROJECT")

In [ ]:
facility = pd.read_csv("Europe_DC_2024_with_facility_id.csv")

# Remove geometry text column if present
facility = facility.drop(columns=["geometry"], errors="ignore")

# Make coordinates numeric
facility["lat"] = pd.to_numeric(facility["lat"], errors="coerce")
facility["lon"] = pd.to_numeric(facility["lon"], errors="coerce")

# Remove invalid coordinates
facility = facility.dropna(subset=["lat", "lon"]).copy()

# Keep only fields needed for Earth Engine
facility_ee_input = facility[[
    "facility_id",
    "country",
    "operator",
    "lat",
    "lon"
]].copy()

facility_ee_input["operator"] = facility_ee_input["operator"].fillna("Unknown").astype(str)

print("Facilities:", len(facility_ee_input))
facility_ee_input.head()

In [ ]:
points_gdf = gpd.GeoDataFrame(
    facility_ee_input,
    geometry=gpd.points_from_xy(
        facility_ee_input["lon"],
        facility_ee_input["lat"]
    ),
    crs="EPSG:4326"
)

try:
    points_ee = geemap.geopandas_to_ee(points_gdf)
except AttributeError:
    points_ee = geemap.gdf_to_ee(points_gdf)

print("Point FeatureCollection created.")
print("Number of points:", points_ee.size().getInfo())

In [ ]:
buffer_distances_m = ee.List([1000, 2500, 5000, 10000])

def make_buffers_for_distance(distance_m):
    distance_m = ee.Number(distance_m)
    buffer_km = distance_m.divide(1000)

    return points_ee.map(
        lambda feature: ee.Feature(
            feature.geometry().buffer(distance_m, maxError=50),
            {
                "facility_id": feature.get("facility_id"),
                "country": feature.get("country"),
                "operator": feature.get("operator"),
                "buffer_m": distance_m,
                "buffer_km": buffer_km
            }
        )
    )

buffers_ee = ee.FeatureCollection(
    buffer_distances_m.map(make_buffers_for_distance)
).flatten()

print("Number of Earth Engine buffer features:", buffers_ee.size().getInfo())

In [ ]:
# Dynamic World classes:
# 0 = water
# 1 = trees
# 2 = grass
# 3 = flooded vegetation
# 4 = crops
# 5 = shrub and scrub
# 6 = built
# 7 = bare
# 8 = snow and ice

dw_2024 = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterDate("2024-01-01", "2025-01-01")
    .filterBounds(points_ee)
)

print("Dynamic World 2024 collection loaded.")

In [ ]:
dw_label_2024 = (
    dw_2024
    .select("label")
    .reduce(ee.Reducer.mode())
    .rename("dw_label")
)

In [ ]:
# Built-up land
dw_built = dw_label_2024.eq(6).rename("dw_built").toFloat()

# Vegetation: trees, grass, flooded vegetation, crops, shrub/scrub
dw_vegetation = (
    dw_label_2024.eq(1)
    .Or(dw_label_2024.eq(2))
    .Or(dw_label_2024.eq(3))
    .Or(dw_label_2024.eq(4))
    .Or(dw_label_2024.eq(5))
    .rename("dw_vegetation")
    .toFloat()
)

# Water
dw_water = dw_label_2024.eq(0).rename("dw_water").toFloat()

# Bare ground
dw_bare = dw_label_2024.eq(7).rename("dw_bare").toFloat()

# Snow and ice
dw_snow_ice = dw_label_2024.eq(8).rename("dw_snow_ice").toFloat()

dw_landcover_indicators = ee.Image.cat([
    dw_built,
    dw_vegetation,
    dw_water,
    dw_bare,
    dw_snow_ice
])

In [ ]:
dw_stats = dw_landcover_indicators.reduceRegions(
    collection=buffers_ee,
    reducer=ee.Reducer.mean(),
    scale=100,
    tileScale=8
)

In [ ]:
def value_to_pct(feature, property_name):
    value = feature.get(property_name)
    return ee.Algorithms.If(
        ee.Algorithms.IsEqual(value, None),
        None,
        ee.Number(value).multiply(100)
    )

def make_dw_table_row(feature):
    return ee.Feature(None, {
        "facility_id": feature.get("facility_id"),
        "country": feature.get("country"),
        "operator": feature.get("operator"),
        "buffer_m": feature.get("buffer_m"),
        "buffer_km": feature.get("buffer_km"),
        "dw_built_pct": value_to_pct(feature, "dw_built"),
        "dw_vegetation_pct": value_to_pct(feature, "dw_vegetation"),
        "dw_water_pct": value_to_pct(feature, "dw_water"),
        "dw_bare_pct": value_to_pct(feature, "dw_bare"),
        "dw_snow_ice_pct": value_to_pct(feature, "dw_snow_ice")
    })

dw_table = dw_stats.map(make_dw_table_row)

In [ ]:
task_dw = ee.batch.Export.table.toDrive(
    collection=dw_table,
    description="DynamicWorld_2024_landcover_by_buffer_FINAL",
    fileNamePrefix="DynamicWorld_2024_landcover_by_buffer_FINAL",
    fileFormat="CSV",
    selectors=[
        "facility_id",
        "country",
        "operator",
        "buffer_m",
        "buffer_km",
        "dw_built_pct",
        "dw_vegetation_pct",
        "dw_water_pct",
        "dw_bare_pct",
        "dw_snow_ice_pct"
    ]
)

task_dw.start()

print("Dynamic World 2024 land-cover export started.")

In [ ]:
task_dw.status()

In [ ]:
drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive")

matches = sorted(
    drive_root.rglob("DynamicWorld_2024_landcover_by_buffer_FINAL*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

matches

In [ ]:
dw_file = matches[0]
print("Using Dynamic World file:", dw_file)

dw_long = pd.read_csv(dw_file)

dw_long["buffer_m"] = pd.to_numeric(dw_long["buffer_m"], errors="coerce")
dw_long["buffer_km"] = pd.to_numeric(dw_long["buffer_km"], errors="coerce")

dw_cols = [
    "dw_built_pct",
    "dw_vegetation_pct",
    "dw_water_pct",
    "dw_bare_pct",
    "dw_snow_ice_pct"
]

for col in dw_cols:
    dw_long[col] = pd.to_numeric(dw_long[col], errors="coerce")

print(dw_long.shape)
dw_long.head()

In [ ]:
print("Rows by buffer:")
print(dw_long["buffer_m"].value_counts().sort_index())

print("\nMissing values:")
print(dw_long[dw_cols].isna().sum())

print("\nSummary:")
display(dw_long[dw_cols].describe().T)

In [ ]:
bad_dw = dw_long[
    (dw_long[dw_cols] < 0).any(axis=1) |
    (dw_long[dw_cols] > 100).any(axis=1) |
    (dw_long[dw_cols].isna()).any(axis=1)
]

print("Bad Dynamic World rows:", len(bad_dw))
bad_dw.head()

In [ ]:
dw_long["dw_total_pct"] = (
    dw_long["dw_built_pct"] +
    dw_long["dw_vegetation_pct"] +
    dw_long["dw_water_pct"] +
    dw_long["dw_bare_pct"] +
    dw_long["dw_snow_ice_pct"]
)

dw_long["dw_total_pct"].describe()

In [ ]:
dw_wide_parts = []

for value_col in dw_cols:
    temp = dw_long.pivot_table(
        index="facility_id",
        columns="buffer_m",
        values=value_col,
        aggfunc="first"
    ).reset_index()

    rename_dict = {
        1000: value_col.replace("_pct", "_pct_1km"),
        2500: value_col.replace("_pct", "_pct_2p5km"),
        5000: value_col.replace("_pct", "_pct_5km"),
        10000: value_col.replace("_pct", "_pct_10km")
    }

    temp = temp.rename(columns=rename_dict)
    dw_wide_parts.append(temp)

dw_wide = dw_wide_parts[0]

for temp in dw_wide_parts[1:]:
    dw_wide = dw_wide.merge(temp, on="facility_id", how="left")

dw_wide.head()

In [ ]:
master = pd.read_csv(
    "Europe_DC_2024_with_population_builtup_indicators_FINAL.csv"
)

master_dw = master.merge(
    dw_wide,
    on="facility_id",
    how="left"
)

print("Rows:", len(master_dw))

new_dw_cols = [c for c in master_dw.columns if c.startswith("dw_")]

print("\nMissing Dynamic World values:")
print(master_dw[new_dw_cols].isna().sum())

print("\nSummary:")
display(master_dw[new_dw_cols].describe().T)

In [ ]:
dw_long.to_csv(
    "Europe_DC_2024_DynamicWorld_landcover_indicators_long_FINAL.csv",
    index=False
)

dw_wide.to_csv(
    "Europe_DC_2024_DynamicWorld_landcover_indicators_FINAL.csv",
    index=False
)

dw_wide.to_excel(
    "Europe_DC_2024_DynamicWorld_landcover_indicators_FINAL.xlsx",
    index=False
)

master_dw.to_csv(
    "Europe_DC_2024_with_population_builtup_DynamicWorld_landcover_FINAL.csv",
    index=False
)

master_dw.to_excel(
    "Europe_DC_2024_with_population_builtup_DynamicWorld_landcover_FINAL.xlsx",
    index=False
)

print("Saved final population + built-up + Dynamic World land-cover outputs.")

In [ ]:
dw_pct_cols = [
    "dw_built_pct_1km", "dw_vegetation_pct_1km", "dw_water_pct_1km", "dw_bare_pct_1km", "dw_snow_ice_pct_1km",
    "dw_built_pct_2p5km", "dw_vegetation_pct_2p5km", "dw_water_pct_2p5km", "dw_bare_pct_2p5km", "dw_snow_ice_pct_2p5km",
    "dw_built_pct_5km", "dw_vegetation_pct_5km", "dw_water_pct_5km", "dw_bare_pct_5km", "dw_snow_ice_pct_5km",
    "dw_built_pct_10km", "dw_vegetation_pct_10km", "dw_water_pct_10km", "dw_bare_pct_10km", "dw_snow_ice_pct_10km"
]

print("Missing values:")
print(master_dw[dw_pct_cols].isna().sum())

print("\nValues below 0:")
print((master_dw[dw_pct_cols] < 0).sum())

print("\nValues above 100:")
print((master_dw[dw_pct_cols] > 100).sum())

# Hotmaps heat demand density

In [ ]:
"""# Hotmaps 2015 heat-demand density"""

!pip -q install rasterio rasterstats geopandas pandas openpyxl requests

import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterstats import zonal_stats
from pathlib import Path
from google.colab import drive
import requests

In [ ]:
drive.mount("/content/drive")

hotmaps_dir = Path("/content/drive/MyDrive/ECHO_DC/hotmaps_heat_2015")
hotmaps_dir.mkdir(parents=True, exist_ok=True)

hotmaps_raster = hotmaps_dir / "heat_tot_curr_density.tif"

url = "https://gitlab.com/hotmaps/buildings/heat/heat_tot_curr_density/-/raw/master/data/heat_tot_curr_density.tif?inline=false"

print("Downloading Hotmaps raster to:")
print(hotmaps_raster)

with requests.get(url, stream=True, allow_redirects=True) as r:
    r.raise_for_status()
    with open(hotmaps_raster, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Download complete.")
print("File size MB:", hotmaps_raster.stat().st_size / 1e6)

In [ ]:
# checking the raster opens correctly
with rasterio.open(hotmaps_raster) as src:
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("NoData:", src.nodata)
    print("Width, height:", src.width, src.height)
    print("Band count:", src.count)

In [ ]:
# loading buffer
buffers = gpd.read_file(
    "Europe_DC_2024_all_buffers.gpkg",
    layer="all_buffers"
)

print("Buffer rows:", len(buffers))
print("Unique facilities:", buffers["facility_id"].nunique())
print("Buffer distances:", sorted(buffers["buffer_km"].unique()))
print("Buffer CRS:", buffers.crs)

In [ ]:
# reproject buffers to match hotmaps

with rasterio.open(hotmaps_raster) as src:
    raster_crs = src.crs
    raster_nodata = src.nodata

buffers_for_hotmaps = buffers.to_crs(raster_crs)

print("Raster CRS:", raster_crs)
print("Buffer CRS after reprojection:", buffers_for_hotmaps.crs)

In [ ]:
# Extract Hotmaps heat demand inside each buffer

zs = zonal_stats(
    buffers_for_hotmaps,
    str(hotmaps_raster),
    raster_out=True,
    nodata=raster_nodata,
    all_touched=False
)

heat_sums = []
heat_means = []
heat_counts = []

for z in zs:
    arr = z["mini_raster_array"]

    # Convert to masked array and remove invalid values
    arr = np.ma.array(arr, dtype="float64")

    # Mask non-finite values
    arr = np.ma.masked_where(~np.isfinite(arr), arr)

    # Mask negative values, because heat demand should not be negative
    arr = np.ma.masked_where(arr < 0, arr)

    if arr.count() == 0:
        heat_sums.append(np.nan)
        heat_means.append(np.nan)
        heat_counts.append(0)
    else:
        heat_sums.append(float(arr.sum()))
        heat_means.append(float(arr.mean()))
        heat_counts.append(int(arr.count()))

In [ ]:
# Create the long Hotmaps table
hotmaps_long = buffers[[
    "facility_id",
    "country",
    "operator",
    "buffer_km"
]].copy()

hotmaps_long["hotmaps_heat_sum"] = heat_sums
hotmaps_long["hotmaps_heat_mean"] = heat_means
hotmaps_long["hotmaps_pixel_count"] = heat_counts

hotmaps_long.head()

In [ ]:
# QA checks
print("Rows:", len(hotmaps_long))

print("\nRows by buffer:")
print(hotmaps_long["buffer_km"].value_counts().sort_index())

print("\nMissing heat values:")
print(hotmaps_long[["hotmaps_heat_sum", "hotmaps_heat_mean"]].isna().sum())

print("\nNegative values:")
print((hotmaps_long[["hotmaps_heat_sum", "hotmaps_heat_mean"]] < 0).sum())

print("\nPixel count summary:")
print(hotmaps_long["hotmaps_pixel_count"].describe())

print("\nHeat-demand summary:")
display(hotmaps_long[["hotmaps_heat_sum", "hotmaps_heat_mean"]].describe().T)

In [ ]:
# Pivot to wide format
hotmaps_sum_wide = hotmaps_long.pivot_table(
    index="facility_id",
    columns="buffer_km",
    values="hotmaps_heat_sum",
    aggfunc="first"
).reset_index()

hotmaps_sum_wide = hotmaps_sum_wide.rename(columns={
    1.0: "hotmaps_heat_sum_1km",
    2.5: "hotmaps_heat_sum_2p5km",
    5.0: "hotmaps_heat_sum_5km",
    10.0: "hotmaps_heat_sum_10km"
})

hotmaps_mean_wide = hotmaps_long.pivot_table(
    index="facility_id",
    columns="buffer_km",
    values="hotmaps_heat_mean",
    aggfunc="first"
).reset_index()

hotmaps_mean_wide = hotmaps_mean_wide.rename(columns={
    1.0: "hotmaps_heat_mean_1km",
    2.5: "hotmaps_heat_mean_2p5km",
    5.0: "hotmaps_heat_mean_5km",
    10.0: "hotmaps_heat_mean_10km"
})

hotmaps_wide = hotmaps_sum_wide.merge(
    hotmaps_mean_wide,
    on="facility_id",
    how="left"
)

hotmaps_wide.head()

In [ ]:
# Join to master table
master = pd.read_csv(
    "Europe_DC_2024_with_population_builtup_DynamicWorld_landcover_FINAL.csv"
)

master_hotmaps = master.merge(
    hotmaps_wide,
    on="facility_id",
    how="left"
)

print("Rows:", len(master_hotmaps))

hotmaps_cols = [
    "hotmaps_heat_sum_1km",
    "hotmaps_heat_sum_2p5km",
    "hotmaps_heat_sum_5km",
    "hotmaps_heat_sum_10km",
    "hotmaps_heat_mean_1km",
    "hotmaps_heat_mean_2p5km",
    "hotmaps_heat_mean_5km",
    "hotmaps_heat_mean_10km"
]

print("\nMissing Hotmaps values:")
print(master_hotmaps[hotmaps_cols].isna().sum())

print("\nNegative Hotmaps values:")
print((master_hotmaps[hotmaps_cols] < 0).sum())

print("\nSummary:")
display(master_hotmaps[hotmaps_cols].describe().T)

In [ ]:
# Save final Hotmaps outputs
hotmaps_long.to_csv(
    "Europe_DC_2024_Hotmaps_heat_demand_indicators_long_FINAL.csv",
    index=False
)

hotmaps_wide.to_csv(
    "Europe_DC_2024_Hotmaps_heat_demand_indicators_FINAL.csv",
    index=False
)

hotmaps_wide.to_excel(
    "Europe_DC_2024_Hotmaps_heat_demand_indicators_FINAL.xlsx",
    index=False
)

master_hotmaps.to_csv(
    "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_FINAL.csv",
    index=False
)

master_hotmaps.to_excel(
    "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_FINAL.xlsx",
    index=False
)

print("Saved final population + built-up + Dynamic World + Hotmaps outputs.")

In [ ]:
# missing sites
hotmaps_cols = [
    "hotmaps_heat_sum_1km",
    "hotmaps_heat_sum_2p5km",
    "hotmaps_heat_sum_5km",
    "hotmaps_heat_sum_10km",
    "hotmaps_heat_mean_1km",
    "hotmaps_heat_mean_2p5km",
    "hotmaps_heat_mean_5km",
    "hotmaps_heat_mean_10km"
]

missing_hotmaps = master_hotmaps[
    master_hotmaps[hotmaps_cols].isna().any(axis=1)
].copy()

print("Facilities with any missing Hotmaps values:", len(missing_hotmaps))

missing_hotmaps[[
    "facility_id",
    "country",
    "name",
    "operator",
    "lat",
    "lon",
    "hotmaps_heat_sum_1km",
    "hotmaps_heat_sum_2p5km",
    "hotmaps_heat_sum_5km",
    "hotmaps_heat_sum_10km",
    "hotmaps_heat_mean_1km",
    "hotmaps_heat_mean_2p5km",
    "hotmaps_heat_mean_5km",
    "hotmaps_heat_mean_10km"
]].head(20)

In [ ]:
missing_hotmaps["country"].value_counts()

In [ ]:
hotmaps_fill_cols = [
    "hotmaps_heat_sum_1km",
    "hotmaps_heat_sum_2p5km",
    "hotmaps_heat_mean_1km",
    "hotmaps_heat_mean_2p5km"
]

master_hotmaps[hotmaps_fill_cols] = master_hotmaps[hotmaps_fill_cols].fillna(0)

print("Missing Hotmaps values after filling:")
print(master_hotmaps[hotmaps_cols].isna().sum())

In [ ]:
print("Negative Hotmaps values:")
print((master_hotmaps[hotmaps_cols] < 0).sum())

print("\nSummary after filling:")
display(master_hotmaps[hotmaps_cols].describe().T)

In [ ]:
#Q&A

master_hotmaps["hotmaps_small_buffer_missing_originally"] = (
    master_hotmaps["facility_id"].isin(missing_hotmaps["facility_id"])
)

master_hotmaps["hotmaps_small_buffer_missing_originally"].value_counts()

In [ ]:
master_hotmaps.to_csv(
    "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_FINAL.csv",
    index=False
)

master_hotmaps.to_excel(
    "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_FINAL.xlsx",
    index=False
)

print("Saved corrected final Hotmaps master table.")

# ERA5-Land heating degree days

In [ ]:
"""# ERA5-Land 2024 Heating Degree Days"""

!pip -q install earthengine-api geemap geopandas pandas openpyxl

import pandas as pd
import geopandas as gpd
import numpy as np
import ee
import geemap
from pathlib import Path
from google.colab import drive

In [ ]:
ee.Authenticate()
ee.Initialize(project="YOUR-EARTH-ENGINE-PROJECT")

In [ ]:
# Loading facility table

facility = pd.read_csv("Europe_DC_2024_with_facility_id.csv")

# Remove geometry text column if present
facility = facility.drop(columns=["geometry"], errors="ignore")

# Make coordinates numeric
facility["lat"] = pd.to_numeric(facility["lat"], errors="coerce")
facility["lon"] = pd.to_numeric(facility["lon"], errors="coerce")

# Remove invalid coordinates
facility = facility.dropna(subset=["lat", "lon"]).copy()

facility_ee_input = facility[[
    "facility_id",
    "country",
    "operator",
    "lat",
    "lon"
]].copy()

facility_ee_input["operator"] = facility_ee_input["operator"].fillna("Unknown").astype(str)

print("Facilities:", len(facility_ee_input))
facility_ee_input.head()

In [ ]:
# Convert facility points to Earth Engine

points_gdf = gpd.GeoDataFrame(
    facility_ee_input,
    geometry=gpd.points_from_xy(
        facility_ee_input["lon"],
        facility_ee_input["lat"]
    ),
    crs="EPSG:4326"
)

try:
    points_ee = geemap.geopandas_to_ee(points_gdf)
except AttributeError:
    points_ee = geemap.gdf_to_ee(points_gdf)

print("Point FeatureCollection created.")
print("Number of points:", points_ee.size().getInfo())

In [ ]:
# Load ERA5-Land daily temperature for 2024

era5_2024 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .filterDate("2024-01-01", "2025-01-01")
    .select("temperature_2m")
)

print("ERA5-Land daily images in 2024:", era5_2024.size().getInfo())

In [ ]:
def calculate_daily_hdd(image):
    # Convert Kelvin to Celsius
    temp_c = image.select("temperature_2m").subtract(273.15)

    # Eurostat-style HDD:
    # if Tmean <= 15°C, HDD = 18°C - Tmean
    # otherwise HDD = 0
    hdd_daily = ee.Image(18).subtract(temp_c).where(
        temp_c.gt(15),
        0
    ).rename("hdd_daily")

    return hdd_daily.copyProperties(image, ["system:time_start"])

hdd_daily_collection = era5_2024.map(calculate_daily_hdd)

hdd_2024 = hdd_daily_collection.sum().rename("hdd_2024")

In [ ]:
# ERA5-Land is coarse, so use a 15 km climate buffer around each facility
# instead of sampling the exact point pixel.

def make_climate_buffer(feature):
    return ee.Feature(
        feature.geometry().buffer(15000, maxError=500),
        {
            "facility_id": feature.get("facility_id"),
            "country": feature.get("country"),
            "operator": feature.get("operator")
        }
    )

climate_buffers_ee = points_ee.map(make_climate_buffer)

print("Climate buffer features:", climate_buffers_ee.size().getInfo())

In [ ]:
# Calculate annual mean 2 m air temperature in Celsius
# This is mainly for QA / interpretation alongside HDD
era5_mean_temp_2024_c = (
    era5_2024.mean()
    .subtract(273.15)
    .rename("era5_mean_temp_2024_c")
)

# Combine annual HDD and annual mean temperature into one Earth Engine image
hdd_image = ee.Image.cat([
    hdd_2024,
    era5_mean_temp_2024_c
])

print("hdd_image created with bands:")
print(hdd_image.bandNames().getInfo())

In [ ]:
hdd_stats = hdd_image.reduceRegions(
    collection=climate_buffers_ee,
    reducer=ee.Reducer.mean(),
    scale=11132,
    tileScale=8
)

In [ ]:
def make_hdd_table_row(feature):
    return ee.Feature(None, {
        "facility_id": feature.get("facility_id"),
        "country": feature.get("country"),
        "operator": feature.get("operator"),
        "hdd_2024": feature.get("hdd_2024"),
        "era5_mean_temp_2024_c": feature.get("era5_mean_temp_2024_c")
    })

hdd_table = hdd_stats.map(make_hdd_table_row)

In [ ]:
task_hdd_buffer = ee.batch.Export.table.toDrive(
    collection=hdd_table,
    description="ERA5Land_HDD_2024_15km_buffer_FINAL",
    fileNamePrefix="ERA5Land_HDD_2024_15km_buffer_FINAL",
    fileFormat="CSV",
    selectors=[
        "facility_id",
        "country",
        "operator",
        "hdd_2024",
        "era5_mean_temp_2024_c"
    ]
)

task_hdd_buffer.start()

print("ERA5-Land HDD 15 km buffer export started.")

In [ ]:
task_hdd_buffer.status()

In [ ]:
# loading new file
drive_root = Path("/content/drive/MyDrive")

matches = sorted(
    drive_root.rglob("ERA5Land_HDD_2024_15km_buffer_FINAL*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

matches

In [ ]:
hdd_file = matches[0]
print("Using HDD file:", hdd_file)

hdd = pd.read_csv(hdd_file)

hdd["hdd_2024"] = pd.to_numeric(hdd["hdd_2024"], errors="coerce")
hdd["era5_mean_temp_2024_c"] = pd.to_numeric(
    hdd["era5_mean_temp_2024_c"],
    errors="coerce"
)

print("Rows:", len(hdd))

print("\nMissing values:")
print(hdd[["hdd_2024", "era5_mean_temp_2024_c"]].isna().sum())

display(hdd[["hdd_2024", "era5_mean_temp_2024_c"]].describe().T)

In [ ]:
# joining to master table

master = pd.read_csv(
    "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_FINAL.csv"
)

hdd_keep = hdd[[
    "facility_id",
    "hdd_2024",
    "era5_mean_temp_2024_c"
]].copy()

master_hdd = master.merge(
    hdd_keep,
    on="facility_id",
    how="left"
)

print("Rows:", len(master_hdd))

print("\nMissing HDD values after join:")
print(master_hdd[["hdd_2024", "era5_mean_temp_2024_c"]].isna().sum())

display(master_hdd[["hdd_2024", "era5_mean_temp_2024_c"]].describe().T)

In [ ]:
hdd.to_csv(
    "Europe_DC_2024_ERA5Land_HDD_2024_15km_buffer_FINAL.csv",
    index=False
)

hdd.to_excel(
    "Europe_DC_2024_ERA5Land_HDD_2024_15km_buffer_FINAL.xlsx",
    index=False
)

master_hdd.to_csv(
    "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_FINAL.csv",
    index=False
)

master_hdd.to_excel(
    "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_FINAL.xlsx",
    index=False
)

print("Saved final master table with ERA5-Land HDD.")

# OSM Anchor Loads - Geofabrik PBF method

In [ ]:
!apt-get -qq update
!apt-get -qq install -y osmium-tool

!pip -q install pandas geopandas shapely pyogrio fiona openpyxl

import pandas as pd
import geopandas as gpd
import numpy as np
import json
import shutil
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

print("Setup complete.")

In [ ]:
# setting up folders

BASE_DIR = Path(".")
DRIVE_DIR = Path("/content/drive/MyDrive/ECHO_DC")

PBF_DIR = DRIVE_DIR / "osm_pbf"
ANCHOR_OUTPUT_DIR = DRIVE_DIR / "osm_anchor_loads_geofabrik"

PBF_DIR.mkdir(parents=True, exist_ok=True)
ANCHOR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DC_CSV = BASE_DIR / "Europe_DC_2024_with_facility_id.csv"
INPUT_BUFFERS_GPKG = BASE_DIR / "Europe_DC_2024_all_buffers.gpkg"
INPUT_POINTS_GPKG = BASE_DIR / "Europe_DC_2024_points_with_facility_id.gpkg"
INPUT_MASTER_CSV = BASE_DIR / "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_FINAL.csv"

print("PBF folder:", PBF_DIR)
print("Anchor output folder:", ANCHOR_OUTPUT_DIR)

for f in [INPUT_DC_CSV, INPUT_BUFFERS_GPKG, INPUT_POINTS_GPKG, INPUT_MASTER_CSV]:
    print(f, "exists:", f.exists())

In [ ]:
#download small test country
PBF_URLS = {
    "IE_NI": "https://download.geofabrik.de/europe/ireland-and-northern-ireland-latest.osm.pbf",
    "PT": "https://download.geofabrik.de/europe/portugal-latest.osm.pbf",
    "GB": "https://download.geofabrik.de/europe/united-kingdom-latest.osm.pbf",
}

In [ ]:
country_code = "IE_NI"

url = PBF_URLS[country_code]
out_file = PBF_DIR / f"{country_code}.osm.pbf"

if out_file.exists():
    print("Already downloaded:", out_file)
else:
    print("Downloading:", url)
    !wget -O "{out_file}" "{url}"

print("File exists:", out_file.exists())
print("File size GB:", out_file.stat().st_size / 1e9)

In [ ]:
# filtering pbf for major anchor loads

def run_command(command):
    print("\nRunning command:")
    print(" ".join(command))

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    print("STDOUT:")
    print(result.stdout[-1000:])

    print("STDERR:")
    print(result.stderr[-1000:])

    if result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}")

    return result

In [ ]:
ANCHOR_FILTERS = [
    "n/amenity=hospital", "w/amenity=hospital", "r/amenity=hospital",
    "n/healthcare=hospital", "w/healthcare=hospital", "r/healthcare=hospital",

    "n/amenity=college", "w/amenity=college", "r/amenity=college",
    "n/amenity=university", "w/amenity=university", "r/amenity=university",

    "n/amenity=nursing_home", "w/amenity=nursing_home", "r/amenity=nursing_home",
    "n/social_facility=nursing_home", "w/social_facility=nursing_home", "r/social_facility=nursing_home",
    "n/social_facility=assisted_living", "w/social_facility=assisted_living", "r/social_facility=assisted_living",
    "n/social_facility=group_home", "w/social_facility=group_home", "r/social_facility=group_home",

    "n/leisure=sports_centre", "w/leisure=sports_centre", "r/leisure=sports_centre",
]

In [ ]:
country_code = "IE_NI"

input_pbf = PBF_DIR / f"{country_code}.osm.pbf"
filtered_pbf = ANCHOR_OUTPUT_DIR / f"{country_code}_anchor_loads_filtered.osm.pbf"

cmd = [
    "osmium", "tags-filter",
    str(input_pbf),
    *ANCHOR_FILTERS,
    "-o", str(filtered_pbf),
    "--overwrite"
]

run_command(cmd)

print("Filtered PBF exists:", filtered_pbf.exists())
print("Filtered size MB:", filtered_pbf.stat().st_size / 1e6)

In [ ]:
country_code = "IE_NI"

filtered_pbf = ANCHOR_OUTPUT_DIR / f"{country_code}_anchor_loads_filtered.osm.pbf"
anchor_geojson = ANCHOR_OUTPUT_DIR / f"{country_code}_anchor_loads.geojson"

cmd = [
    "osmium", "export",
    str(filtered_pbf),
    "-o", str(anchor_geojson),
    "--overwrite"
]

run_command(cmd)

print("GeoJSON exists:", anchor_geojson.exists())
print("GeoJSON size MB:", anchor_geojson.stat().st_size / 1e6)

In [ ]:
# load and classifying test anchors

def classify_anchor(row):
    amenity = str(row.get("amenity", ""))
    healthcare = str(row.get("healthcare", ""))
    leisure = str(row.get("leisure", ""))
    social_facility = str(row.get("social_facility", ""))

    if amenity == "hospital" or healthcare == "hospital":
        return "healthcare"

    if amenity in ["college", "university"]:
        return "education"

    if amenity == "nursing_home" or social_facility in [
        "nursing_home", "assisted_living", "group_home"
    ]:
        return "care"

    if leisure == "sports_centre":
        return "leisure"

    return "other"

In [ ]:
country_code = "IE_NI"

anchor_geojson = ANCHOR_OUTPUT_DIR / f"{country_code}_anchor_loads.geojson"

anchors = gpd.read_file(anchor_geojson)

print("Raw features:", len(anchors))
print("Columns:", anchors.columns.tolist())
print("Geometry types:")
print(anchors.geometry.geom_type.value_counts())

anchors["source_country_file"] = country_code
anchors["anchor_category"] = anchors.apply(classify_anchor, axis=1)

anchors = anchors[anchors["anchor_category"] != "other"].copy()

print("\nAnchor categories:")
print(anchors["anchor_category"].value_counts())

anchors.head()

In [ ]:
# Keep only useful geometry types for anchor-load facilities
valid_geom_types = ["Point", "Polygon", "MultiPolygon"]

print("Before geometry filtering:")
print(anchors.geometry.geom_type.value_counts())

anchors_clean = anchors[
    anchors.geometry.geom_type.isin(valid_geom_types)
].copy()

print("\nAfter geometry filtering:")
print(anchors_clean.geometry.geom_type.value_counts())

print("\nAnchor categories after geometry filtering:")
print(anchors_clean["anchor_category"].value_counts())

In [ ]:
# Keep only the columns needed for the indicators
useful_cols = [
    "source_country_file",
    "anchor_category",
    "name",
    "amenity",
    "healthcare",
    "leisure",
    "social_facility",
    "operator",
    "beds",
    "capacity",
    "geometry"
]

# Keep only columns that actually exist in this country file
useful_cols = [c for c in useful_cols if c in anchors_clean.columns]

anchors_clean = anchors_clean[useful_cols].copy()

anchors_clean.head()

In [ ]:
# converting cleaned anchors to points

def anchors_to_points(anchors_gdf):
    anchors_3035 = anchors_gdf.to_crs("EPSG:3035").copy()

    # For polygons, representative_point gives a point guaranteed to lie inside the polygon.
    # For existing points, it just returns the same point.
    anchors_3035["geometry"] = anchors_3035.geometry.representative_point()

    anchors_points = anchors_3035.to_crs("EPSG:4326")

    return anchors_points

In [ ]:
anchors_points = anchors_to_points(anchors_clean)

print("Point geometries:")
print(anchors_points.geometry.geom_type.value_counts())

anchors_points.head()

In [ ]:
# creating an anchor id
anchors_points["x_round"] = anchors_points.geometry.x.round(6)
anchors_points["y_round"] = anchors_points.geometry.y.round(6)

anchors_points["name_clean"] = (
    anchors_points["name"]
    .fillna("unnamed")
    .astype(str)
    .str.lower()
    .str.strip()
)

anchors_points["anchor_id"] = (
    anchors_points["source_country_file"].astype(str)
    + "_"
    + anchors_points["anchor_category"].astype(str)
    + "_"
    + anchors_points["name_clean"].astype(str)
    + "_"
    + anchors_points["x_round"].astype(str)
    + "_"
    + anchors_points["y_round"].astype(str)
)

print("Rows before deduplication:", len(anchors_points))

anchors_points = anchors_points.drop_duplicates(subset=["anchor_id"]).copy()

print("Rows after deduplication:", len(anchors_points))

print("\nFinal anchor categories:")
print(anchors_points["anchor_category"].value_counts())

In [ ]:
# removing helper columns

anchors_points = anchors_points.drop(
    columns=["x_round", "y_round", "name_clean"],
    errors="ignore"
)

anchors_points.head()

In [ ]:
# saving the cleaned test file
test_anchor_csv = ANCHOR_OUTPUT_DIR / "IE_NI_anchor_loads_points_TEST.csv"
test_anchor_gpkg = ANCHOR_OUTPUT_DIR / "IE_NI_anchor_loads_points_TEST.gpkg"

anchors_points.to_csv(test_anchor_csv, index=False)

anchors_points.to_file(
    test_anchor_gpkg,
    layer="anchor_loads",
    driver="GPKG"
)

print("Saved:")
print(test_anchor_csv)
print(test_anchor_gpkg)

Full country loop

In [ ]:
PBF_URLS = {
    "NL": "https://download.geofabrik.de/europe/netherlands-latest.osm.pbf",
    "DE": "https://download.geofabrik.de/europe/germany-latest.osm.pbf",
    "FR": "https://download.geofabrik.de/europe/france-latest.osm.pbf",
    "GB": "https://download.geofabrik.de/europe/united-kingdom-latest.osm.pbf",
    "IE_NI": "https://download.geofabrik.de/europe/ireland-and-northern-ireland-latest.osm.pbf",
    "BE": "https://download.geofabrik.de/europe/belgium-latest.osm.pbf",
    "ES": "https://download.geofabrik.de/europe/spain-latest.osm.pbf",
    "IT": "https://download.geofabrik.de/europe/italy-latest.osm.pbf",
    "PL": "https://download.geofabrik.de/europe/poland-latest.osm.pbf",
    "SE": "https://download.geofabrik.de/europe/sweden-latest.osm.pbf",
    "FI": "https://download.geofabrik.de/europe/finland-latest.osm.pbf",
    "NO": "https://download.geofabrik.de/europe/norway-latest.osm.pbf",
    "DK": "https://download.geofabrik.de/europe/denmark-latest.osm.pbf",
    "AT": "https://download.geofabrik.de/europe/austria-latest.osm.pbf",
    "CH": "https://download.geofabrik.de/europe/switzerland-latest.osm.pbf",
    "PT": "https://download.geofabrik.de/europe/portugal-latest.osm.pbf",
    "CZ": "https://download.geofabrik.de/europe/czech-republic-latest.osm.pbf",
    "HU": "https://download.geofabrik.de/europe/hungary-latest.osm.pbf",
    "RO": "https://download.geofabrik.de/europe/romania-latest.osm.pbf",
    "GR": "https://download.geofabrik.de/europe/greece-latest.osm.pbf",
}

In [ ]:
COUNTRIES_TO_PROCESS = [
    "NL", "DE", "FR", "GB", "IE_NI", "BE", "ES", "IT", "PL", "SE",
    "FI", "NO", "DK", "AT", "CH", "PT", "CZ", "HU", "RO", "GR"
]

In [ ]:
# downloading and filtering to combine anchors

def download_pbf(country_code):
    url = PBF_URLS[country_code]
    out_file = PBF_DIR / f"{country_code}.osm.pbf"

    if out_file.exists():
        print(f"{country_code}: PBF already exists.")
        return out_file

    print(f"{country_code}: downloading from {url}")
    cmd = ["wget", "-O", str(out_file), url]
    run_command(cmd)

    return out_file


def filter_anchor_pbf(country_code, input_pbf):
    filtered_pbf = ANCHOR_OUTPUT_DIR / f"{country_code}_anchor_loads_filtered.osm.pbf"

    if filtered_pbf.exists():
        print(f"{country_code}: filtered PBF already exists.")
        return filtered_pbf

    cmd = [
        "osmium", "tags-filter",
        str(input_pbf),
        *ANCHOR_FILTERS,
        "-o", str(filtered_pbf),
        "--overwrite"
    ]

    run_command(cmd)

    return filtered_pbf


def export_anchor_geojson(country_code, filtered_pbf):
    anchor_geojson = ANCHOR_OUTPUT_DIR / f"{country_code}_anchor_loads.geojson"

    if anchor_geojson.exists():
        print(f"{country_code}: GeoJSON already exists.")
        return anchor_geojson

    cmd = [
        "osmium", "export",
        str(filtered_pbf),
        "-o", str(anchor_geojson),
        "--overwrite"
    ]

    run_command(cmd)

    return anchor_geojson


def load_anchor_geojson(country_code, anchor_geojson):
    anchors = gpd.read_file(anchor_geojson)

    if len(anchors) == 0:
        return anchors

    anchors["source_country_file"] = country_code
    anchors["anchor_category"] = anchors.apply(classify_anchor, axis=1)
    anchors = anchors[anchors["anchor_category"] != "other"].copy()

    anchors_points = anchors_to_points(anchors)

    return anchors_points

In [ ]:
anchors_all = gpd.read_file(
    "/content/drive/MyDrive/ECHO_DC/osm_anchor_loads_geofabrik/Europe_OSM_major_anchor_loads_geofabrik_CLEAN.gpkg",
    layer="anchor_loads"
)

In [ ]:
from pathlib import Path

ANCHOR_OUTPUT_DIR = Path("/content/drive/MyDrive/ECHO_DC/osm_anchor_loads_geofabrik")

print("Folder exists:", ANCHOR_OUTPUT_DIR.exists())
print("Folder path:", ANCHOR_OUTPUT_DIR)

files = sorted(ANCHOR_OUTPUT_DIR.glob("*CLEAN*"))

print("Number of CLEAN files:", len(files))

for f in files[:50]:
    print(f.name)

In [ ]:
#!ls -lh "/content/drive/MyDrive/ECHO_DC/osm_anchor_loads_geofabrik" | head -80

In [ ]:
combined_files = [
    "Europe_OSM_major_anchor_loads_geofabrik_CLEAN.csv",
    "Europe_OSM_major_anchor_loads_geofabrik_CLEAN.gpkg",
    "Europe_OSM_major_anchor_loads_geofabrik_CLEAN.geojson"
]

for filename in combined_files:
    f = ANCHOR_OUTPUT_DIR / filename
    print(filename, "exists:", f.exists(), "size MB:", f.stat().st_size / 1e6 if f.exists() else None)

In [ ]:
import pandas as pd

dc = pd.read_csv("Europe_DC_2024_with_facility_id.csv")

dc_countries = sorted(dc["country"].dropna().unique())
anchor_countries = sorted(anchors_all["source_country_file"].dropna().unique())

print("Data-centre countries:")
print(dc_countries)

print("\nAnchor countries:")
print(anchor_countries)

print("\nCountries in data centres but missing from anchors:")
print(sorted(set(dc_countries) - set(anchor_countries)))

In [ ]:
# turning IE_NI to IE

# Standardise source country code for reporting
anchors_all["source_country_code"] = anchors_all["source_country_file"].replace({
    "IE_NI": "IE"
})

print("Anchor source country codes after standardising:")
print(sorted(anchors_all["source_country_code"].unique()))

print("\nCountries in data centres but missing from anchors after standardising:")
print(sorted(set(dc_countries) - set(anchors_all["source_country_code"].unique())))

In [ ]:
# Remove possible duplicate anchors that appear in overlapping Geofabrik extracts
# especially Northern Ireland, which can appear in both GB and IE_NI extracts.

anchors_all["x_round"] = anchors_all.geometry.x.round(6)
anchors_all["y_round"] = anchors_all.geometry.y.round(6)

anchors_all["name_clean"] = (
    anchors_all["name"]
    .fillna("unnamed")
    .astype(str)
    .str.lower()
    .str.strip()
)

anchors_all["cross_source_anchor_id"] = (
    anchors_all["anchor_category"].astype(str)
    + "_"
    + anchors_all["name_clean"].astype(str)
    + "_"
    + anchors_all["x_round"].astype(str)
    + "_"
    + anchors_all["y_round"].astype(str)
)

print("Rows before cross-source deduplication:", len(anchors_all))

anchors_all = anchors_all.drop_duplicates(
    subset=["cross_source_anchor_id"],
    keep="first"
).copy()

print("Rows after cross-source deduplication:", len(anchors_all))

anchors_all = anchors_all.drop(
    columns=["x_round", "y_round", "name_clean", "cross_source_anchor_id"],
    errors="ignore"
)

print("\nCategory counts after final deduplication:")
print(anchors_all["anchor_category"].value_counts())

print("\nCountry-code counts after final deduplication:")
print(anchors_all["source_country_code"].value_counts().sort_index())

In [ ]:
# resaving final version

anchors_all_csv = ANCHOR_OUTPUT_DIR / "Europe_OSM_major_anchor_loads_geofabrik_CLEAN_DEDUPED.csv"
anchors_all_gpkg = ANCHOR_OUTPUT_DIR / "Europe_OSM_major_anchor_loads_geofabrik_CLEAN_DEDUPED.gpkg"
anchors_all_geojson = ANCHOR_OUTPUT_DIR / "Europe_OSM_major_anchor_loads_geofabrik_CLEAN_DEDUPED.geojson"

if anchors_all_gpkg.exists():
    anchors_all_gpkg.unlink()

anchors_all.to_csv(anchors_all_csv, index=False)

anchors_all.to_file(
    anchors_all_gpkg,
    layer="anchor_loads",
    driver="GPKG"
)

anchors_all.to_file(
    anchors_all_geojson,
    driver="GeoJSON"
)

print("Saved final deduplicated anchor files:")
print(anchors_all_csv)
print(anchors_all_gpkg)
print(anchors_all_geojson)

# Mapping anchor loads

In [ ]:
# loading files
!pip -q install geopandas folium mapclassify

import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster, FastMarkerCluster
from pathlib import Path

BASE_DIR = Path(".")
ANCHOR_OUTPUT_DIR = Path("/content/drive/MyDrive/ECHO_DC/osm_anchor_loads_geofabrik")

DC_POINTS_FILE = BASE_DIR / "Europe_DC_2024_points_with_facility_id.gpkg"
BUFFERS_FILE = BASE_DIR / "Europe_DC_2024_all_buffers.gpkg"

ANCHORS_FILE = ANCHOR_OUTPUT_DIR / "Europe_OSM_major_anchor_loads_geofabrik_CLEAN_DEDUPED.gpkg"

print("DC points exists:", DC_POINTS_FILE.exists())
print("Buffers exists:", BUFFERS_FILE.exists())
print("Anchors exists:", ANCHORS_FILE.exists())

In [ ]:
# reading

dc_points = gpd.read_file(
    DC_POINTS_FILE,
    layer="data_centres"
).to_crs("EPSG:4326")

buffers = gpd.read_file(
    BUFFERS_FILE,
    layer="all_buffers"
)

anchors = gpd.read_file(
    ANCHORS_FILE,
    layer="anchor_loads"
).to_crs("EPSG:4326")

print("DC sites:", len(dc_points))
print("Buffers:", len(buffers))
print("Anchors:", len(anchors))

print("\nAnchor categories:")
print(anchors["anchor_category"].value_counts())

In [ ]:
# clipping anchors to 10km of DCs

# Select only the 10 km buffers
buffers_10km = buffers[buffers["buffer_km"] == 10].copy()

print("10 km buffers:", len(buffers_10km))

# Reproject for spatial join
anchors_3035 = anchors.to_crs(buffers_10km.crs)

# Join anchors that fall within any 10 km DC buffer
anchors_near_dc = gpd.sjoin(
    anchors_3035,
    buffers_10km[["facility_id", "geometry"]],
    how="inner",
    predicate="within"
)

# Remove duplicates where one anchor falls inside several overlapping DC buffers
anchors_near_dc = anchors_near_dc.drop_duplicates(subset=["anchor_id"]).copy()

# Back to WGS84 for Folium
anchors_near_dc = anchors_near_dc.to_crs("EPSG:4326")

print("Anchors within 10 km of at least one DC:", len(anchors_near_dc))

print("\nCategory counts near DCs:")
print(anchors_near_dc["anchor_category"].value_counts())

In [ ]:
# creating map

# Centre map roughly on Europe
m = folium.Map(
    location=[53, 10],
    zoom_start=4,
    tiles=None
)

# Satellite basemap
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite basemap",
    overlay=False,
    control=True
).add_to(m)

# Optional labels layer
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_only_labels/{z}/{x}/{y}{r}.png",
    attr="CartoDB labels",
    name="Labels",
    overlay=True,
    control=True
).add_to(m)

m

In [ ]:
# adding DC locations

dc_cluster = MarkerCluster(name="Data centres").add_to(m)

for _, row in dc_points.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x

    site_name = row.get("site_label", row.get("name", row.get("facility_id", "Data centre")))
    operator = row.get("operator_label", row.get("operator", ""))

    popup_html = f"""
    <b>Data centre</b><br>
    <b>Facility ID:</b> {row.get("facility_id", "")}<br>
    <b>Name:</b> {site_name}<br>
    <b>Operator:</b> {operator}<br>
    <b>Country:</b> {row.get("country", "")}
    """

    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(dc_cluster)

m

In [ ]:
# Data centres = red
dc_cluster = MarkerCluster(name="Data centres").add_to(m)

for _, row in dc_points.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x

    site_name = row.get("site_label", row.get("name", row.get("facility_id", "Data centre")))
    operator = row.get("operator_label", row.get("operator", ""))

    popup_html = f"""
    <b>Data centre</b><br>
    <b>Facility ID:</b> {row.get("facility_id", "")}<br>
    <b>Name:</b> {site_name}<br>
    <b>Operator:</b> {operator}<br>
    <b>Country:</b> {row.get("country", "")}
    """

    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.95,
        weight=1,
        popup=folium.Popup(popup_html, max_width=300)
    ).add_to(dc_cluster)

m

In [ ]:
# Anchor loads = cyan/blue
anchor_data = [
    [geom.y, geom.x]
    for geom in anchors_near_dc.geometry
]

anchor_layer = folium.FeatureGroup(
    name="OSM anchor loads",
    show=True
)

anchor_callback = """
function (row) {
    return L.circleMarker(new L.LatLng(row[0], row[1]), {
        radius: 3,
        color: 'cyan',
        fillColor: 'cyan',
        fillOpacity: 0.65,
        weight: 1
    });
}
"""

FastMarkerCluster(
    data=anchor_data,
    callback=anchor_callback
).add_to(anchor_layer)

anchor_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
map_file = ANCHOR_OUTPUT_DIR / "Europe_DC_locations_with_OSM_anchor_loads_two_colour_satellite_map.html"

m.save(map_file)

print("Saved map to:")
print(map_file)

Connecting anchor dataset to DC dataset

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

BASE_DIR = Path(".")
ANCHOR_OUTPUT_DIR = Path("/content/drive/MyDrive/ECHO_DC/osm_anchor_loads_geofabrik")

ANCHORS_FILE = ANCHOR_OUTPUT_DIR / "Europe_OSM_major_anchor_loads_geofabrik_CLEAN_DEDUPED.gpkg"

BUFFERS_FILE = BASE_DIR / "Europe_DC_2024_all_buffers.gpkg"
POINTS_FILE = BASE_DIR / "Europe_DC_2024_points_with_facility_id.gpkg"
MASTER_FILE = BASE_DIR / "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_FINAL.csv"

print("Anchors exists:", ANCHORS_FILE.exists())
print("Buffers exists:", BUFFERS_FILE.exists())
print("Points exists:", POINTS_FILE.exists())
print("Master exists:", MASTER_FILE.exists())

In [ ]:
# loading anchors, DCs and buffers
anchors = gpd.read_file(
    ANCHORS_FILE,
    layer="anchor_loads"
).to_crs("EPSG:4326")

buffers = gpd.read_file(
    BUFFERS_FILE,
    layer="all_buffers"
)

dc_points = gpd.read_file(
    POINTS_FILE,
    layer="data_centres"
).to_crs("EPSG:4326")

master = pd.read_csv(MASTER_FILE)

print("Anchors:", len(anchors))
print("Buffers:", len(buffers))
print("DC points:", len(dc_points))
print("Master rows:", len(master))

print("\nAnchor CRS:", anchors.crs)
print("Buffer CRS:", buffers.crs)
print("DC points CRS:", dc_points.crs)

print("\nAnchor categories:")
print(anchors["anchor_category"].value_counts())

print("\nBuffer distances:")
print(sorted(buffers["buffer_km"].unique()))

In [ ]:
# checks before joining

print("Missing anchor IDs:", anchors["anchor_id"].isna().sum())
print("Duplicate anchor IDs:", anchors["anchor_id"].duplicated().sum())
print("Missing anchor geometry:", anchors.geometry.isna().sum())
print("Empty anchor geometry:", anchors.geometry.is_empty.sum())

print("\nMissing buffer facility IDs:", buffers["facility_id"].isna().sum())
print("Missing DC facility IDs:", dc_points["facility_id"].isna().sum())
print("Missing master facility IDs:", master["facility_id"].isna().sum())

print("\nUnique facilities in buffers:", buffers["facility_id"].nunique())
print("Unique facilities in points:", dc_points["facility_id"].nunique())
print("Unique facilities in master:", master["facility_id"].nunique())

In [ ]:
# reprojecting to metric CRS

# Make sure buffer_km is numeric
buffers["buffer_km"] = pd.to_numeric(buffers["buffer_km"], errors="coerce")

# Ensure everything spatial is in EPSG:3035 for joining/distances
buffers_3035 = buffers.to_crs("EPSG:3035")
anchors_3035 = anchors.to_crs("EPSG:3035")
dc_points_3035 = dc_points.to_crs("EPSG:3035")

print("Buffers CRS:", buffers_3035.crs)
print("Anchors CRS:", anchors_3035.crs)
print("DC CRS:", dc_points_3035.crs)

In [ ]:
# filtering anchors to near DCs

buffers_10km = buffers_3035[buffers_3035["buffer_km"] == 10].copy()

print("10 km buffers:", len(buffers_10km))

anchors_near_dc_join = gpd.sjoin(
    anchors_3035,
    buffers_10km[["facility_id", "geometry"]],
    how="inner",
    predicate="within"
)

print("Anchor-buffer matches at 10 km:", len(anchors_near_dc_join))

# Keep each anchor once for the next full buffer join
near_anchor_ids = anchors_near_dc_join["anchor_id"].drop_duplicates()

anchors_near_dc = anchors_3035[
    anchors_3035["anchor_id"].isin(near_anchor_ids)
].copy()

print("Unique anchors within 10 km of at least one DC:", len(anchors_near_dc))

print("\nCategories near DCs:")
print(anchors_near_dc["anchor_category"].value_counts())

In [ ]:
# joining anchors to all buffers

buffers_keep = buffers_3035[[
    "facility_id",
    "country",
    "operator",
    "buffer_km",
    "geometry"
]].copy()

anchor_buffer_join = gpd.sjoin(
    anchors_near_dc,
    buffers_keep,
    how="inner",
    predicate="within"
)

print("Total anchor-buffer join rows:", len(anchor_buffer_join))

print("\nBuffer-distance counts in join:")
print(anchor_buffer_join["buffer_km"].value_counts().sort_index())

anchor_buffer_join.head()

In [ ]:
# total anchor counts per buffer
facility_ids = master[["facility_id"]].drop_duplicates().copy()

anchor_counts_long = (
    anchor_buffer_join
    .groupby(["facility_id", "buffer_km"])
    .size()
    .reset_index(name="anchor_count")
)

anchor_counts_wide = anchor_counts_long.pivot_table(
    index="facility_id",
    columns="buffer_km",
    values="anchor_count",
    aggfunc="first",
    fill_value=0
).reset_index()

anchor_counts_wide = anchor_counts_wide.rename(columns={
    1.0: "anchor_count_1km",
    2.5: "anchor_count_2p5km",
    5.0: "anchor_count_5km",
    10.0: "anchor_count_10km"
})

anchor_counts_wide = facility_ids.merge(
    anchor_counts_wide,
    on="facility_id",
    how="left"
)

anchor_count_cols = [
    "anchor_count_1km",
    "anchor_count_2p5km",
    "anchor_count_5km",
    "anchor_count_10km"
]

for col in anchor_count_cols:
    if col not in anchor_counts_wide.columns:
        anchor_counts_wide[col] = 0

anchor_counts_wide[anchor_count_cols] = (
    anchor_counts_wide[anchor_count_cols]
    .fillna(0)
    .astype(int)
)

print("Rows:", len(anchor_counts_wide))
print(anchor_counts_wide[anchor_count_cols].describe().T)

anchor_counts_wide.head()

In [ ]:
# category specific anchor counts

categories = ["healthcare", "education", "care", "leisure"]

category_counts_long = (
    anchor_buffer_join
    .groupby(["facility_id", "buffer_km", "anchor_category"])
    .size()
    .reset_index(name="count")
)

category_counts_wide = facility_ids.copy()

for category in categories:
    temp = category_counts_long[
        category_counts_long["anchor_category"] == category
    ].pivot_table(
        index="facility_id",
        columns="buffer_km",
        values="count",
        aggfunc="first",
        fill_value=0
    ).reset_index()

    temp = temp.rename(columns={
        1.0: f"{category}_anchor_count_1km",
        2.5: f"{category}_anchor_count_2p5km",
        5.0: f"{category}_anchor_count_5km",
        10.0: f"{category}_anchor_count_10km"
    })

    category_counts_wide = category_counts_wide.merge(
        temp,
        on="facility_id",
        how="left"
    )

category_anchor_cols = [
    c for c in category_counts_wide.columns
    if c != "facility_id"
]

category_counts_wide[category_anchor_cols] = (
    category_counts_wide[category_anchor_cols]
    .fillna(0)
    .astype(int)
)

print("Rows:", len(category_counts_wide))
category_counts_wide.head()

In [ ]:
# anchor type diversity

anchor_diversity_long = (
    anchor_buffer_join
    .groupby(["facility_id", "buffer_km"])["anchor_category"]
    .nunique()
    .reset_index(name="anchor_type_diversity")
)

anchor_diversity_wide = anchor_diversity_long.pivot_table(
    index="facility_id",
    columns="buffer_km",
    values="anchor_type_diversity",
    aggfunc="first",
    fill_value=0
).reset_index()

anchor_diversity_wide = anchor_diversity_wide.rename(columns={
    1.0: "anchor_type_diversity_1km",
    2.5: "anchor_type_diversity_2p5km",
    5.0: "anchor_type_diversity_5km",
    10.0: "anchor_type_diversity_10km"
})

anchor_diversity_cols = [
    "anchor_type_diversity_1km",
    "anchor_type_diversity_2p5km",
    "anchor_type_diversity_5km",
    "anchor_type_diversity_10km"
]

anchor_diversity_wide = facility_ids.merge(
    anchor_diversity_wide,
    on="facility_id",
    how="left"
)

anchor_diversity_wide[anchor_diversity_cols] = (
    anchor_diversity_wide[anchor_diversity_cols]
    .fillna(0)
    .astype(int)
)

print("Rows:", len(anchor_diversity_wide))
anchor_diversity_wide.head()

In [ ]:
# nearest anchor distance

anchors_nearest = anchors_3035[[
    "anchor_id",
    "anchor_category",
    "name",
    "source_country_code",
    "geometry"
]].rename(columns={
    "anchor_id": "nearest_anchor_id",
    "anchor_category": "nearest_anchor_category",
    "name": "nearest_anchor_name",
    "source_country_code": "nearest_anchor_source_country"
})

nearest_anchor = gpd.sjoin_nearest(
    dc_points_3035[["facility_id", "geometry"]],
    anchors_nearest,
    how="left",
    distance_col="nearest_anchor_distance_m"
)

nearest_anchor = nearest_anchor[[
    "facility_id",
    "nearest_anchor_distance_m",
    "nearest_anchor_id",
    "nearest_anchor_category",
    "nearest_anchor_name",
    "nearest_anchor_source_country"
]]

# If there are tied nearest anchors, keep one
nearest_anchor = (
    nearest_anchor
    .sort_values(["facility_id", "nearest_anchor_distance_m"])
    .drop_duplicates(subset=["facility_id"], keep="first")
    .reset_index(drop=True)
)

print("Rows:", len(nearest_anchor))
print("Missing nearest distances:", nearest_anchor["nearest_anchor_distance_m"].isna().sum())

nearest_anchor["nearest_anchor_distance_m"].describe()

In [ ]:
# combining all anchor indicators

anchor_indicators = (
    anchor_counts_wide
    .merge(category_counts_wide, on="facility_id", how="left")
    .merge(anchor_diversity_wide, on="facility_id", how="left")
    .merge(nearest_anchor, on="facility_id", how="left")
)

print("Rows:", len(anchor_indicators))
print("Unique facilities:", anchor_indicators["facility_id"].nunique())

anchor_indicators.head()

In [ ]:
# checks

anchor_indicators["anchor_count_monotonic_ok"] = (
    (anchor_indicators["anchor_count_1km"] <= anchor_indicators["anchor_count_2p5km"]) &
    (anchor_indicators["anchor_count_2p5km"] <= anchor_indicators["anchor_count_5km"]) &
    (anchor_indicators["anchor_count_5km"] <= anchor_indicators["anchor_count_10km"])
)

print("Monotonic check:")
print(anchor_indicators["anchor_count_monotonic_ok"].value_counts())

print("\nMissing values:")
print(anchor_indicators.isna().sum().sort_values(ascending=False).head(20))

print("\nAnchor count summary:")
display(anchor_indicators[anchor_count_cols].describe().T)

print("\nDiversity summary:")
display(anchor_indicators[anchor_diversity_cols].describe().T)


In [ ]:
# joining anchor indicators to main table

master_anchors = master.merge(
    anchor_indicators,
    on="facility_id",
    how="left"
)

print("Master rows:", len(master_anchors))
print("Unique facilities:", master_anchors["facility_id"].nunique())

anchor_cols = [c for c in master_anchors.columns if "anchor" in c]

print("\nMissing anchor-related values:")
print(master_anchors[anchor_cols].isna().sum().sort_values(ascending=False).head(30))


In [ ]:
# fill count and diversity columns with zero

count_and_diversity_cols = [
    c for c in master_anchors.columns
    if c.endswith("_count_1km")
    or c.endswith("_count_2p5km")
    or c.endswith("_count_5km")
    or c.endswith("_count_10km")
    or c.startswith("anchor_type_diversity_")
]

count_and_diversity_cols += [
    "anchor_count_1km",
    "anchor_count_2p5km",
    "anchor_count_5km",
    "anchor_count_10km"
]

count_and_diversity_cols = list(dict.fromkeys(count_and_diversity_cols))

master_anchors[count_and_diversity_cols] = (
    master_anchors[count_and_diversity_cols]
    .fillna(0)
    .astype(int)
)

print("Filled count/diversity columns with zero.")

In [ ]:
# saving outputs

anchor_indicators_csv = ANCHOR_OUTPUT_DIR / "Europe_DC_2024_OSM_anchor_load_indicators_FINAL.csv"
anchor_indicators_xlsx = ANCHOR_OUTPUT_DIR / "Europe_DC_2024_OSM_anchor_load_indicators_FINAL.xlsx"

master_anchors_csv = "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_Anchors_FINAL.csv"
master_anchors_xlsx = "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_Anchors_FINAL.xlsx"

anchor_indicators.to_csv(anchor_indicators_csv, index=False)
anchor_indicators.to_excel(anchor_indicators_xlsx, index=False)

master_anchors.to_csv(master_anchors_csv, index=False)
master_anchors.to_excel(master_anchors_xlsx, index=False)

print("Saved:")
print(anchor_indicators_csv)
print(anchor_indicators_xlsx)
print(master_anchors_csv)
print(master_anchors_xlsx)

In [ ]:
# final check

print("Final master table rows:", len(master_anchors))
print("Final master unique facilities:", master_anchors["facility_id"].nunique())

selected_cols = [
    "facility_id",
    "country",
    "anchor_count_1km",
    "anchor_count_2p5km",
    "anchor_count_5km",
    "anchor_count_10km",
    "healthcare_anchor_count_2p5km",
    "education_anchor_count_2p5km",
    "care_anchor_count_2p5km",
    "leisure_anchor_count_2p5km",
    "anchor_type_diversity_2p5km",
    "nearest_anchor_distance_m",
    "nearest_anchor_category",
    "nearest_anchor_name"
]

selected_cols = [c for c in selected_cols if c in master_anchors.columns]

display(master_anchors[selected_cols].head(20))

# Accesability variables



In [ ]:
# setup

"""# OSM Accessibility Variables - memory-safe strategic roads method"""

!apt-get -qq update
!apt-get -qq install -y osmium-tool

!pip -q install pandas geopandas shapely pyogrio fiona openpyxl

import pandas as pd
import geopandas as gpd
import numpy as np
import subprocess
import gc
from pathlib import Path

BASE_DIR = Path(".")
DRIVE_DIR = Path("/content/drive/MyDrive/ECHO_DC")

PBF_DIR = DRIVE_DIR / "osm_pbf"

# Use a new folder so it does not mix with the previous crashed attempt
ACCESS_OUTPUT_DIR = DRIVE_DIR / "osm_accessibility_strategic_roads"
ACCESS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POINTS_FILE = BASE_DIR / "Europe_DC_2024_points_with_facility_id.gpkg"
BUFFERS_FILE = BASE_DIR / "Europe_DC_2024_all_buffers.gpkg"
MASTER_FILE = BASE_DIR / "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_Anchors_FINAL.csv"

print("PBF folder exists:", PBF_DIR.exists())
print("Output folder:", ACCESS_OUTPUT_DIR)

for f in [POINTS_FILE, BUFFERS_FILE, MASTER_FILE]:
    print(f, "exists:", f.exists())

In [ ]:
# countries and road classes

COUNTRIES_TO_PROCESS = [
    "NL", "DE", "FR", "GB", "IE_NI", "BE", "ES", "IT", "PL", "SE",
    "FI", "NO", "DK", "AT", "CH", "PT", "CZ", "HU", "RO", "GR"
]

# Strategic roads only: lighter and more defensible for accessibility
ROAD_CLASSES = [
    "motorway",
    "motorway_link",
    "trunk",
    "trunk_link",
    "primary",
    "primary_link"
]

ROAD_FILTERS = []

for road_class in ROAD_CLASSES:
    ROAD_FILTERS.append(f"w/highway={road_class}")
    ROAD_FILTERS.append(f"r/highway={road_class}")

ROAD_FILTERS

In [ ]:
# command helper

def run_command(command):
    print("\nRunning command:")
    print(" ".join(command))

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    print("STDOUT:")
    print(result.stdout[-1000:])

    print("STDERR:")
    print(result.stderr[-1000:])

    if result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}")

    return result

In [ ]:
# loading 10km DC buffers

buffers = gpd.read_file(
    BUFFERS_FILE,
    layer="all_buffers"
)

master = pd.read_csv(MASTER_FILE)

# Make sure country is available in the buffers
if "country" not in buffers.columns:
    buffers = buffers.merge(
        master[["facility_id", "country"]].drop_duplicates(),
        on="facility_id",
        how="left"
    )

buffers["buffer_km"] = pd.to_numeric(buffers["buffer_km"], errors="coerce")

buffers_10km = buffers[buffers["buffer_km"] == 10].copy()
buffers_10km_3035 = buffers_10km.to_crs("EPSG:3035")

print("10 km buffers:", len(buffers_10km_3035))
print("Countries in buffers:")
print(sorted(buffers_10km_3035["country"].dropna().unique()))

In [ ]:
# exporting roads per country

def filter_strategic_road_pbf(country_code):
    input_pbf = PBF_DIR / f"{country_code}.osm.pbf"
    filtered_pbf = ACCESS_OUTPUT_DIR / f"{country_code}_strategic_roads_filtered.osm.pbf"

    if not input_pbf.exists():
        raise FileNotFoundError(f"Missing PBF file: {input_pbf}")

    if filtered_pbf.exists():
        print(f"{country_code}: filtered strategic roads PBF already exists.")
        return filtered_pbf

    cmd = [
        "osmium", "tags-filter",
        str(input_pbf),
        *ROAD_FILTERS,
        "-o", str(filtered_pbf),
        "--overwrite"
    ]

    run_command(cmd)

    return filtered_pbf


def export_strategic_road_geojson(country_code, filtered_pbf):
    road_geojson = ACCESS_OUTPUT_DIR / f"{country_code}_strategic_roads.geojson"

    if road_geojson.exists():
        print(f"{country_code}: strategic road GeoJSON already exists.")
        return road_geojson

    cmd = [
        "osmium", "export",
        str(filtered_pbf),
        "-o", str(road_geojson),
        "--overwrite"
    ]

    run_command(cmd)

    return road_geojson

In [ ]:
# cleaning and clipping one country at a time

def read_roads_minimal(road_geojson):
    """
    Read only useful road columns where possible.
    This reduces memory compared with loading every OSM tag column.
    """
    wanted_cols = ["highway", "name", "ref", "operator", "geometry"]

    try:
        roads = gpd.read_file(
            road_geojson,
            columns=wanted_cols,
            engine="pyogrio"
        )
    except Exception as e:
        print("Minimal read failed, using normal read. Error:", e)
        roads = gpd.read_file(road_geojson)

    return roads


def process_country_roads_near_dc(country_code):
    print("\n==============================")
    print("Processing strategic roads:", country_code)
    print("==============================")

    # IE_NI is the Geofabrik extract name, but your DC table uses IE
    dc_country_code = "IE" if country_code == "IE_NI" else country_code

    country_buffers = buffers_10km_3035[
        buffers_10km_3035["country"] == dc_country_code
    ].copy()

    if len(country_buffers) == 0:
        print(f"No data-centre buffers for {country_code} / {dc_country_code}. Skipping.")
        return None

    filtered_pbf = filter_strategic_road_pbf(country_code)
    road_geojson = export_strategic_road_geojson(country_code, filtered_pbf)

    roads = read_roads_minimal(road_geojson)

    print("Raw road features:", len(roads))

    if len(roads) == 0:
        return None

    # Keep suitable road geometries only
    roads = roads[
        roads.geometry.notna()
        & ~roads.geometry.is_empty
        & roads.geometry.geom_type.isin(["LineString", "MultiLineString"])
    ].copy()

    print("After geometry filtering:", len(roads))

    # Ensure useful columns exist
    useful_cols = [
        "highway",
        "name",
        "ref",
        "operator",
        "geometry"
    ]

    for col in useful_cols:
        if col not in roads.columns and col != "geometry":
            roads[col] = ""

    roads = roads[useful_cols].copy()

    roads["highway"] = roads["highway"].fillna("").astype(str)

    roads = roads[
        roads["highway"].isin(ROAD_CLASSES)
    ].copy()

    print("After road-class filtering:", len(roads))
    print(roads["highway"].value_counts())

    if len(roads) == 0:
        return None

    roads["source_country_file"] = country_code
    roads["source_country_code"] = dc_country_code

    # Safe text fields
    for col in roads.columns:
        if col != "geometry":
            roads[col] = (
                roads[col]
                .fillna("")
                .astype(str)
                .str.slice(0, 250)
            )

    # Create simple unique road ID
    roads = roads.reset_index(drop=True)
    roads["road_id"] = (
        roads["source_country_file"].astype(str)
        + "_road_"
        + roads.index.astype(str)
    )

    # Reproject to metric CRS
    roads_3035 = roads.to_crs("EPSG:3035")

    print("Finding roads within 10 km of data centres...")

    # Keep only roads intersecting any 10 km DC buffer in this country
    near_join = gpd.sjoin(
        roads_3035,
        country_buffers[["facility_id", "geometry"]],
        how="inner",
        predicate="intersects"
    )

    print("Road-buffer intersect rows:", len(near_join))

    if len(near_join) == 0:
        return None

    roads_near_dc = near_join.drop_duplicates(subset=["road_id"]).copy()

    roads_near_dc = roads_near_dc.drop(
        columns=["index_right", "facility_id"],
        errors="ignore"
    )

    print("Unique strategic roads near DCs:", len(roads_near_dc))

    # Save in WGS84 for portability
    roads_near_dc = roads_near_dc.to_crs("EPSG:4326")

    final_cols = [
        "road_id",
        "source_country_file",
        "source_country_code",
        "highway",
        "name",
        "ref",
        "operator",
        "geometry"
    ]

    roads_near_dc = roads_near_dc[final_cols].copy()

    country_roads_gpkg = ACCESS_OUTPUT_DIR / f"{country_code}_strategic_roads_near_dc_CLEAN.gpkg"

    if country_roads_gpkg.exists():
        country_roads_gpkg.unlink()

    roads_near_dc.to_file(
        country_roads_gpkg,
        layer="strategic_roads",
        driver="GPKG"
    )

    print("Saved:", country_roads_gpkg)

    return country_roads_gpkg

In [ ]:
# country loop

processed_road_files = []
failed_roads = []

for country_code in COUNTRIES_TO_PROCESS:
    try:
        output_file = process_country_roads_near_dc(country_code)

        if output_file is not None:
            processed_road_files.append(output_file)

    except Exception as e:
        print("\nFAILED:", country_code)
        print("Error:", e)

        failed_roads.append({
            "country_code": country_code,
            "error": str(e)
        })

    # Clear memory after every country
    gc.collect()

print("\nCountries with saved road files:", len(processed_road_files))
print("Countries failed:", len(failed_roads))

if len(failed_roads) > 0:
    failed_roads_df = pd.DataFrame(failed_roads)
    display(failed_roads_df)

    failed_roads_df.to_csv(
        ACCESS_OUTPUT_DIR / "failed_strategic_road_processing.csv",
        index=False
    )

In [ ]:
# combining near dc road files

road_files = sorted(ACCESS_OUTPUT_DIR.glob("*_strategic_roads_near_dc_CLEAN.gpkg"))

print("Road files found:", len(road_files))

all_road_gdfs = []

for road_file in road_files:
    print("Loading:", road_file.name)
    gdf = gpd.read_file(road_file, layer="strategic_roads")
    all_road_gdfs.append(gdf)

roads_all = pd.concat(all_road_gdfs, ignore_index=True)

roads_all = gpd.GeoDataFrame(
    roads_all,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Combined near-DC strategic roads:", len(roads_all))
print("\nRoad classes:")
print(roads_all["highway"].value_counts())
print("\nCountries:")
print(roads_all["source_country_code"].value_counts().sort_index())

In [ ]:
# saving
roads_all_gpkg = ACCESS_OUTPUT_DIR / "Europe_OSM_strategic_roads_near_DC_CLEAN.gpkg"
roads_all_csv = ACCESS_OUTPUT_DIR / "Europe_OSM_strategic_roads_near_DC_CLEAN.csv"

if roads_all_gpkg.exists():
    roads_all_gpkg.unlink()

roads_all.to_file(
    roads_all_gpkg,
    layer="strategic_roads",
    driver="GPKG"
)

roads_all.to_csv(roads_all_csv, index=False)

print("Saved:")
print(roads_all_gpkg)
print(roads_all_csv)

In [ ]:
# loading roads, buffers and main dataset

import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

BASE_DIR = Path(".")
DRIVE_DIR = Path("/content/drive/MyDrive/ECHO_DC")

ACCESS_OUTPUT_DIR = DRIVE_DIR / "osm_accessibility_strategic_roads"

ROADS_FILE = ACCESS_OUTPUT_DIR / "Europe_OSM_strategic_roads_near_DC_CLEAN.gpkg"

POINTS_FILE = BASE_DIR / "Europe_DC_2024_points_with_facility_id.gpkg"
BUFFERS_FILE = BASE_DIR / "Europe_DC_2024_all_buffers.gpkg"
MASTER_FILE = BASE_DIR / "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_Anchors_FINAL.csv"

roads_all = gpd.read_file(
    ROADS_FILE,
    layer="strategic_roads"
).to_crs("EPSG:4326")

dc_points = gpd.read_file(
    POINTS_FILE,
    layer="data_centres"
).to_crs("EPSG:4326")

buffers = gpd.read_file(
    BUFFERS_FILE,
    layer="all_buffers"
)

master = pd.read_csv(MASTER_FILE)

print("Roads:", len(roads_all))
print("DC points:", len(dc_points))
print("Buffers:", len(buffers))
print("Master rows:", len(master))

print("\nRoad classes:")
print(roads_all["highway"].value_counts())

In [ ]:
# QA and deduplication

print("Rows before deduplication:", len(roads_all))

roads_all["road_dedup_key"] = (
    roads_all["highway"].fillna("").astype(str)
    + "_"
    + roads_all["name"].fillna("").astype(str)
    + "_"
    + roads_all["ref"].fillna("").astype(str)
    + "_"
    + roads_all.geometry.apply(lambda g: g.wkb_hex)
)

roads_all = roads_all.drop_duplicates(subset=["road_dedup_key"]).copy()
roads_all = roads_all.drop(columns=["road_dedup_key"], errors="ignore")

print("Rows after deduplication:", len(roads_all))

print("\nCountries:")
print(roads_all["source_country_code"].value_counts().sort_index())

print("\nMissing geometries:", roads_all.geometry.isna().sum())
print("Empty geometries:", roads_all.geometry.is_empty.sum())

In [ ]:
# reprojecting to EPSG:3035

buffers["buffer_km"] = pd.to_numeric(buffers["buffer_km"], errors="coerce")

roads_3035 = roads_all.to_crs("EPSG:3035")
dc_points_3035 = dc_points.to_crs("EPSG:3035")
buffers_3035 = buffers.to_crs("EPSG:3035")

print("Road CRS:", roads_3035.crs)
print("DC CRS:", dc_points_3035.crs)
print("Buffer CRS:", buffers_3035.crs)
print("Buffer distances:", sorted(buffers_3035["buffer_km"].unique()))

In [ ]:
# calculating nearest road

roads_nearest = roads_3035[[
    "road_id",
    "highway",
    "name",
    "ref",
    "geometry"
]].rename(columns={
    "road_id": "nearest_major_road_id",
    "highway": "nearest_major_road_class",
    "name": "nearest_major_road_name",
    "ref": "nearest_major_road_ref"
})

nearest_major_road = gpd.sjoin_nearest(
    dc_points_3035[["facility_id", "geometry"]],
    roads_nearest,
    how="left",
    distance_col="nearest_major_road_distance_m"
)

nearest_major_road = nearest_major_road[[
    "facility_id",
    "nearest_major_road_distance_m",
    "nearest_major_road_id",
    "nearest_major_road_class",
    "nearest_major_road_name",
    "nearest_major_road_ref"
]]

nearest_major_road = (
    nearest_major_road
    .sort_values(["facility_id", "nearest_major_road_distance_m"])
    .drop_duplicates(subset=["facility_id"], keep="first")
    .reset_index(drop=True)
)

print("Rows:", len(nearest_major_road))
print("Missing nearest road distances:", nearest_major_road["nearest_major_road_distance_m"].isna().sum())

nearest_major_road["nearest_major_road_distance_m"].describe()

In [ ]:
# road length inside buffers

buffers_keep = buffers_3035[[
    "facility_id",
    "buffer_km",
    "geometry"
]].copy()

buffers_keep = buffers_keep.reset_index(drop=True)
buffers_keep["buffer_id"] = buffers_keep.index

roads_for_join = roads_3035[[
    "road_id",
    "highway",
    "geometry"
]].copy()

print("Roads for join:", len(roads_for_join))
print("Buffers for join:", len(buffers_keep))

In [ ]:
road_buffer_join = gpd.sjoin(
    roads_for_join,
    buffers_keep[["buffer_id", "facility_id", "buffer_km", "geometry"]],
    how="inner",
    predicate="intersects"
)

print("Road-buffer intersect rows:", len(road_buffer_join))
print("\nRows by buffer distance:")
print(road_buffer_join["buffer_km"].value_counts().sort_index())

In [ ]:
buffer_geom_lookup = buffers_keep.set_index("buffer_id")["geometry"]

road_buffer_join["buffer_geometry"] = road_buffer_join["buffer_id"].map(buffer_geom_lookup)

print("Calculating clipped road lengths...")

road_buffer_join["road_length_m_in_buffer"] = [
    road_geom.intersection(buffer_geom).length
    for road_geom, buffer_geom in zip(
        road_buffer_join.geometry,
        road_buffer_join["buffer_geometry"]
    )
]

road_buffer_join["road_length_km_in_buffer"] = (
    road_buffer_join["road_length_m_in_buffer"] / 1000
)

print("Done.")

road_buffer_join[[
    "facility_id",
    "buffer_km",
    "highway",
    "road_length_km_in_buffer"
]].head()

In [ ]:
# road length by DC and buffer

road_length_long = (
    road_buffer_join
    .groupby(["facility_id", "buffer_km"])["road_length_km_in_buffer"]
    .sum()
    .reset_index(name="major_road_length_km")
)

road_length_wide = road_length_long.pivot_table(
    index="facility_id",
    columns="buffer_km",
    values="major_road_length_km",
    aggfunc="first",
    fill_value=0
).reset_index()

road_length_wide = road_length_wide.rename(columns={
    1.0: "major_road_length_1km_km",
    2.5: "major_road_length_2p5km_km",
    5.0: "major_road_length_5km_km",
    10.0: "major_road_length_10km_km"
})

length_cols = [
    "major_road_length_1km_km",
    "major_road_length_2p5km_km",
    "major_road_length_5km_km",
    "major_road_length_10km_km"
]

for col in length_cols:
    if col not in road_length_wide.columns:
        road_length_wide[col] = 0

road_length_wide[length_cols] = road_length_wide[length_cols].fillna(0).round(3)

print("Rows:", len(road_length_wide))
display(road_length_wide[length_cols].describe().T)

road_length_wide.head()

In [ ]:
# convert road length to road density

buffer_area_lookup = (
    buffers_keep
    .assign(buffer_area_km2=buffers_keep.geometry.area / 1_000_000)
    [["facility_id", "buffer_km", "buffer_area_km2"]]
)

buffer_area_wide = buffer_area_lookup.pivot_table(
    index="facility_id",
    columns="buffer_km",
    values="buffer_area_km2",
    aggfunc="first"
).reset_index()

buffer_area_wide = buffer_area_wide.rename(columns={
    1.0: "buffer_area_1km_km2",
    2.5: "buffer_area_2p5km_km2",
    5.0: "buffer_area_5km_km2",
    10.0: "buffer_area_10km_km2"
})

road_accessibility = road_length_wide.merge(
    buffer_area_wide,
    on="facility_id",
    how="left"
)

road_accessibility["major_road_density_1km_km_per_km2"] = (
    road_accessibility["major_road_length_1km_km"] / road_accessibility["buffer_area_1km_km2"]
)

road_accessibility["major_road_density_2p5km_km_per_km2"] = (
    road_accessibility["major_road_length_2p5km_km"] / road_accessibility["buffer_area_2p5km_km2"]
)

road_accessibility["major_road_density_5km_km_per_km2"] = (
    road_accessibility["major_road_length_5km_km"] / road_accessibility["buffer_area_5km_km2"]
)

road_accessibility["major_road_density_10km_km_per_km2"] = (
    road_accessibility["major_road_length_10km_km"] / road_accessibility["buffer_area_10km_km2"]
)

density_cols = [
    "major_road_density_1km_km_per_km2",
    "major_road_density_2p5km_km_per_km2",
    "major_road_density_5km_km_per_km2",
    "major_road_density_10km_km_per_km2"
]

road_accessibility[density_cols] = road_accessibility[density_cols].round(4)

road_accessibility = road_accessibility.merge(
    nearest_major_road,
    on="facility_id",
    how="left"
)

print("Rows:", len(road_accessibility))
display(road_accessibility.head())

In [ ]:
# make sure all present

facility_ids = master[["facility_id"]].drop_duplicates().copy()

road_accessibility = facility_ids.merge(
    road_accessibility,
    on="facility_id",
    how="left"
)

for col in length_cols + density_cols:
    if col not in road_accessibility.columns:
        road_accessibility[col] = 0

road_accessibility[length_cols + density_cols] = (
    road_accessibility[length_cols + density_cols]
    .fillna(0)
)

print("Rows:", len(road_accessibility))
print("Unique facilities:", road_accessibility["facility_id"].nunique())

print("\nMissing nearest road distance:")
print(road_accessibility["nearest_major_road_distance_m"].isna().sum())

print("\nRoad length summary:")
display(road_accessibility[length_cols].describe().T)

print("\nRoad density summary:")
display(road_accessibility[density_cols].describe().T)

In [ ]:
# check

road_accessibility["major_road_length_monotonic_ok"] = (
    (road_accessibility["major_road_length_1km_km"] <= road_accessibility["major_road_length_2p5km_km"]) &
    (road_accessibility["major_road_length_2p5km_km"] <= road_accessibility["major_road_length_5km_km"]) &
    (road_accessibility["major_road_length_5km_km"] <= road_accessibility["major_road_length_10km_km"])
)

print(road_accessibility["major_road_length_monotonic_ok"].value_counts())

In [ ]:
# save indicators

road_accessibility_csv = ACCESS_OUTPUT_DIR / "Europe_DC_2024_OSM_strategic_road_accessibility_indicators_FINAL.csv"
road_accessibility_xlsx = ACCESS_OUTPUT_DIR / "Europe_DC_2024_OSM_strategic_road_accessibility_indicators_FINAL.xlsx"

road_accessibility.to_csv(road_accessibility_csv, index=False)
road_accessibility.to_excel(road_accessibility_xlsx, index=False)

print("Saved:")
print(road_accessibility_csv)
print(road_accessibility_xlsx)

In [ ]:
# joiing to main dataset

master_access = master.merge(
    road_accessibility,
    on="facility_id",
    how="left"
)

print("Master rows:", len(master_access))
print("Unique facilities:", master_access["facility_id"].nunique())

access_cols = [
    c for c in master_access.columns
    if "road" in c.lower()
]

print("\nAccessibility-related missing values:")
print(master_access[access_cols].isna().sum().sort_values(ascending=False).head(30))

display(master_access[[
    "facility_id",
    "country",
    "nearest_major_road_distance_m",
    "nearest_major_road_class",
    "major_road_length_2p5km_km",
    "major_road_density_2p5km_km_per_km2"
]].head(20))

In [ ]:
# saving new dataset

master_access_csv = "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_Anchors_Accessibility_FINAL.csv"
master_access_xlsx = "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_Anchors_Accessibility_FINAL.xlsx"

master_access.to_csv(master_access_csv, index=False)
master_access.to_excel(master_access_xlsx, index=False)

print("Saved:")
print(master_access_csv)
print(master_access_xlsx)

# Data confidence score

In [ ]:
# load data

import pandas as pd
import numpy as np
from pathlib import Path

MASTER_FILE = Path("Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_Anchors_Accessibility_FINAL.csv")

df = pd.read_csv(MASTER_FILE)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique facilities:", df["facility_id"].nunique())

df.head()

In [ ]:
# create metadata confidence flags - check basics are complete

# Coordinates
df["conf_has_coordinates"] = (
    df["lat"].notna()
    & df["lon"].notna()
).astype(int)

# Country
df["conf_has_country"] = (
    df["country"].notna()
    & (df["country"].astype(str).str.strip() != "")
).astype(int)

# Site/data-centre name
df["conf_has_name"] = (
    df["name"].notna()
    & (df["name"].astype(str).str.strip() != "")
    & (df["name"].astype(str).str.lower().str.strip() != "nan")
).astype(int)

# Operator
df["conf_has_operator"] = (
    df["operator"].notna()
    & (df["operator"].astype(str).str.strip() != "")
    & (df["operator"].astype(str).str.lower().str.strip() != "nan")
).astype(int)

# Confirmation / source date field
df["conf_has_confirmed_as_of"] = (
    df["confirmed_as_of"].notna()
    & (df["confirmed_as_of"].astype(str).str.strip() != "")
).astype(int)

metadata_conf_cols = [
    "conf_has_coordinates",
    "conf_has_country",
    "conf_has_name",
    "conf_has_operator",
    "conf_has_confirmed_as_of"
]

print(df[metadata_conf_cols].sum())

In [ ]:
# spatial indicator availibility

# Population
df["conf_has_population_data"] = (
    df[["pop_1km", "pop_2p5km", "pop_5km", "pop_10km"]]
    .notna()
    .all(axis=1)
).astype(int)

# GHSL built-up
df["conf_has_ghsl_builtup_data"] = (
    df[["built_m2_1km", "built_m2_2p5km", "built_m2_5km", "built_m2_10km",
        "built_pct_1km", "built_pct_2p5km", "built_pct_5km", "built_pct_10km"]]
    .notna()
    .all(axis=1)
).astype(int)

# Dynamic World land cover
dynamic_world_cols = [
    c for c in df.columns
    if c.startswith("dw_") and c.endswith(("1km", "2p5km", "5km", "10km"))
]

df["conf_has_dynamicworld_data"] = (
    df[dynamic_world_cols]
    .notna()
    .all(axis=1)
).astype(int)

# Hotmaps heat demand
hotmaps_cols = [
    "hotmaps_heat_sum_1km",
    "hotmaps_heat_sum_2p5km",
    "hotmaps_heat_sum_5km",
    "hotmaps_heat_sum_10km",
    "hotmaps_heat_mean_1km",
    "hotmaps_heat_mean_2p5km",
    "hotmaps_heat_mean_5km",
    "hotmaps_heat_mean_10km"
]

df["conf_has_hotmaps_data"] = (
    df[hotmaps_cols]
    .notna()
    .all(axis=1)
).astype(int)

# ERA5-Land HDD
df["conf_has_hdd_data"] = (
    df["hdd_2024"].notna()
    & df["era5_mean_temp_2024_c"].notna()
).astype(int)

# OSM anchor-load data
anchor_core_cols = [
    "anchor_count_1km",
    "anchor_count_2p5km",
    "anchor_count_5km",
    "anchor_count_10km",
    "nearest_anchor_distance_m"
]

df["conf_has_anchor_data"] = (
    df[anchor_core_cols]
    .notna()
    .all(axis=1)
).astype(int)

# OSM road accessibility data
road_core_cols = [
    "major_road_length_1km_km",
    "major_road_length_2p5km_km",
    "major_road_length_5km_km",
    "major_road_length_10km_km",
    "major_road_density_1km_km_per_km2",
    "major_road_density_2p5km_km_per_km2",
    "major_road_density_5km_km_per_km2",
    "major_road_density_10km_km_per_km2"
]

df["conf_has_accessibility_data"] = (
    df[road_core_cols]
    .notna()
    .all(axis=1)
).astype(int)

# Nearest road is separate because you said one site has a missing nearest road distance
df["conf_has_nearest_road"] = (
    df["nearest_major_road_distance_m"].notna()
).astype(int)

spatial_conf_cols = [
    "conf_has_population_data",
    "conf_has_ghsl_builtup_data",
    "conf_has_dynamicworld_data",
    "conf_has_hotmaps_data",
    "conf_has_hdd_data",
    "conf_has_anchor_data",
    "conf_has_accessibility_data",
    "conf_has_nearest_road"
]

print(df[spatial_conf_cols].sum())

In [ ]:
# QA confidence

# Population monotonic check
df["conf_pop_monotonic_ok"] = (
    df["pop_monotonic_ok"].fillna(False).astype(bool)
).astype(int)

# Anchor count monotonic check
df["conf_anchor_monotonic_ok"] = (
    df["anchor_count_monotonic_ok"].fillna(False).astype(bool)
).astype(int)

# Road length monotonic check
df["conf_road_monotonic_ok"] = (
    df["major_road_length_monotonic_ok"].fillna(False).astype(bool)
).astype(int)

# Hotmaps small-buffer gap
# True in original column means there was a small-buffer issue, so confidence flag should be 0.
df["conf_hotmaps_no_small_buffer_gap"] = (
    df["hotmaps_small_buffer_missing_originally"].fillna(False) == False
).astype(int)

qa_conf_cols = [
    "conf_pop_monotonic_ok",
    "conf_anchor_monotonic_ok",
    "conf_road_monotonic_ok",
    "conf_hotmaps_no_small_buffer_gap"
]

print(df[qa_conf_cols].sum())

In [ ]:
# creating confidence scores,

# Core location confidence: directly affects spatial analysis
core_location_conf_cols = [
    "conf_has_coordinates",
    "conf_has_country",
    "conf_has_confirmed_as_of"
]

# Metadata completeness: useful for documentation, but not part of analytical confidence
metadata_only_conf_cols = [
    "conf_has_name",
    "conf_has_operator"
]

# Spatial indicator confidence: core ECHO-DC evidence base
spatial_conf_cols = [
    "conf_has_population_data",
    "conf_has_ghsl_builtup_data",
    "conf_has_dynamicworld_data",
    "conf_has_hotmaps_data",
    "conf_has_hdd_data",
    "conf_has_anchor_data",
    "conf_has_accessibility_data",
    "conf_has_nearest_road"
]

# QA confidence
qa_conf_cols = [
    "conf_pop_monotonic_ok",
    "conf_anchor_monotonic_ok",
    "conf_road_monotonic_ok",
    "conf_hotmaps_no_small_buffer_gap"
]

df["core_location_confidence_score"] = df[core_location_conf_cols].mean(axis=1)

df["metadata_completeness_score"] = df[metadata_only_conf_cols].mean(axis=1)

df["spatial_data_confidence_score"] = df[spatial_conf_cols].mean(axis=1)

df["qa_confidence_score"] = df[qa_conf_cols].mean(axis=1)

df["analytical_data_confidence_score"] = (
    0.40 * df["core_location_confidence_score"]
    + 0.45 * df["spatial_data_confidence_score"]
    + 0.15 * df["qa_confidence_score"]
)

df["analytical_data_confidence_score"] = (
    df["analytical_data_confidence_score"].round(3)
)

df[[
    "core_location_confidence_score",
    "metadata_completeness_score",
    "spatial_data_confidence_score",
    "qa_confidence_score",
    "analytical_data_confidence_score"
]].describe()

In [ ]:
# classification

def classify_confidence(score):
    if score >= 0.85:
        return "High"
    elif score >= 0.65:
        return "Medium"
    else:
        return "Low"


df["analytical_data_confidence_class"] = (
    df["analytical_data_confidence_score"].apply(classify_confidence)
)

print(df["analytical_data_confidence_class"].value_counts())

display(
    df[[
        "facility_id",
        "country",
        "name",
        "operator",
        "core_location_confidence_score",
        "metadata_completeness_score",
        "spatial_data_confidence_score",
        "qa_confidence_score",
        "analytical_data_confidence_score",
        "analytical_data_confidence_class"
    ]].head(20)
)

In [ ]:
## Check Low confidence records

low_confidence = df[
    df["analytical_data_confidence_class"] == "Low"
].copy()

print("Low-confidence records:", len(low_confidence))

display(
    low_confidence[[
        "facility_id",
        "country",
        "name",
        "operator",
        "core_location_confidence_score",
        "metadata_completeness_score",
        "spatial_data_confidence_score",
        "qa_confidence_score",
        "analytical_data_confidence_score",
        "analytical_data_confidence_class",
        "conf_has_coordinates",
        "conf_has_country",
        "conf_has_confirmed_as_of",
        "conf_has_population_data",
        "conf_has_ghsl_builtup_data",
        "conf_has_dynamicworld_data",
        "conf_has_hotmaps_data",
        "conf_has_hdd_data",
        "conf_has_anchor_data",
        "conf_has_accessibility_data",
        "conf_has_nearest_road",
        "conf_hotmaps_no_small_buffer_gap"
    ]].head(50)
)

In [ ]:
# Check Medium confidence records

medium_confidence = df[
    df["analytical_data_confidence_class"] == "Medium"
].copy()

print("Medium-confidence records:", len(medium_confidence))

display(
    medium_confidence[[
        "facility_id",
        "country",
        "name",
        "operator",
        "core_location_confidence_score",
        "metadata_completeness_score",
        "spatial_data_confidence_score",
        "qa_confidence_score",
        "analytical_data_confidence_score",
        "analytical_data_confidence_class",
        "conf_has_coordinates",
        "conf_has_country",
        "conf_has_confirmed_as_of",
        "conf_has_hotmaps_data",
        "conf_hotmaps_no_small_buffer_gap",
        "conf_has_nearest_road"
    ]].head(50)
)

In [ ]:
# Metadata completeness is reported separately and does not affect analytical confidence

metadata_summary = pd.DataFrame({
    "metadata_field": ["name", "operator"],
    "records_available": [
        df["conf_has_name"].sum(),
        df["conf_has_operator"].sum()
    ],
    "records_missing": [
        len(df) - df["conf_has_name"].sum(),
        len(df) - df["conf_has_operator"].sum()
    ],
    "percent_available": [
        round((df["conf_has_name"].sum() / len(df)) * 100, 2),
        round((df["conf_has_operator"].sum() / len(df)) * 100, 2)
    ]
})

display(metadata_summary)

In [ ]:
print(df["analytical_data_confidence_score"].describe())

print("\nConfidence class counts:")
print(df["analytical_data_confidence_class"].value_counts())

print("\nUnique confidence scores:")
print(df["analytical_data_confidence_score"].value_counts().sort_index())

In [ ]:
# saving

output_csv = "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_Anchors_Accessibility_Confidence_FINAL.csv"
output_xlsx = "Europe_DC_2024_with_population_builtup_DynamicWorld_Hotmaps_HDD_Anchors_Accessibility_Confidence_FINAL.xlsx"

df.to_csv(output_csv, index=False)
df.to_excel(output_xlsx, index=False)

print("Saved:")
print(output_csv)
print(output_xlsx)

print("\nFinal confidence class counts:")
print(df["analytical_data_confidence_class"].value_counts())

print("\nAnalytical confidence score summary:")
display(df["analytical_data_confidence_score"].describe())

In [ ]:
# saving confidence only table

confidence_cols = [
    "facility_id",
    "country",
    "name",
    "operator",

    # Core location confidence used in analytical score
    "conf_has_coordinates",
    "conf_has_country",
    "conf_has_confirmed_as_of",

    # Metadata completeness reported separately
    "conf_has_name",
    "conf_has_operator",

    # Spatial indicator confidence
    "conf_has_population_data",
    "conf_has_ghsl_builtup_data",
    "conf_has_dynamicworld_data",
    "conf_has_hotmaps_data",
    "conf_has_hdd_data",
    "conf_has_anchor_data",
    "conf_has_accessibility_data",
    "conf_has_nearest_road",

    # QA confidence
    "conf_pop_monotonic_ok",
    "conf_anchor_monotonic_ok",
    "conf_road_monotonic_ok",
    "conf_hotmaps_no_small_buffer_gap",

    # Scores
    "core_location_confidence_score",
    "metadata_completeness_score",
    "spatial_data_confidence_score",
    "qa_confidence_score",
    "analytical_data_confidence_score",
    "analytical_data_confidence_class"
]

confidence_table = df[confidence_cols].copy()

confidence_csv = "Europe_DC_2024_data_confidence_indicators_FINAL.csv"
confidence_xlsx = "Europe_DC_2024_data_confidence_indicators_FINAL.xlsx"

confidence_table.to_csv(confidence_csv, index=False)
confidence_table.to_excel(confidence_xlsx, index=False)

print("Saved:")
print(confidence_csv)
print(confidence_xlsx)

display(confidence_table.head(20))